# EfficientNetV2 S / M / L: Akurasi, Efisiensi, dan Kualitas Penjelasan (XAI) — v3

Notebook eksperimen untuk studi: *mengapa dan kapan varian EfficientNetV2 yang lebih besar tidak memberi penjelasan (heatmap) yang lebih tepat terhadap lokasi lesi: karena modelnya, atau karena metode penjelasannya?*
Rujukan metodologi: `desain-eksperimen-opsi-A-v2.md`. Setiap run melatih satu varian pada satu dataset, lalu langsung menghitung performa, skor XAI per citra, uji ketergantungan pada lesi, uji sanity bertingkat, dan ERF.

## Perubahan v3.4 (pembagian plan `full` ke 3 akun)
| Kebutuhan | Perubahan v3.4 |
|---|---|
| Plan `full` (sekitar 64 jam GPU) terlalu lama untuk satu akun | Opsi **`SHARD`** di sel konfigurasi membagi 45 run `full` ke 3 akun **milik 3 orang berbeda** (Terms of Use Kaggle melarang satu orang memakai lebih dari satu akun). Pembagian tetap: akun 1 = HAM10000 seed 1–3; akun 2 = HAM10000 seed 4–5 + BUSI fold 0–4; akun 3 = SIIM seed 1–5 |
| Setiap run harus dikerjakan tepat sekali | Sel 5 memeriksa bahwa pembagian mencakup semua run `full` tepat satu kali, lalu menampilkan run dan perkiraan jam GPU untuk shard ini |
| Asal setiap hasil harus tercatat | `summary.json` setiap run mencatat `shard`; `environment.txt` tersimpan per akun |
| Benchmark efisiensi harus dari satu lingkungan | Sel 7 hanya berjalan bila `SHARD = None` (sesi penggabungan akhir di akun utama) |

## Perubahan v3.3 (setelah `lr_search`)
| Kebutuhan | Perubahan v3.3 |
|---|---|
| 11 kombinasi mendapat LR terbaik di tepi grid {3e-5, 1e-4, 3e-4}. Dokumen desain 4.4 menetapkan bahwa grid diperlebar | Mode baru **`lr_extend`**: untuk setiap kombinasi (dataset, varian, fold) yang LR terbaiknya di tepi bawah, diuji 1e-5; di tepi atas, diuji 1e-3; kalau tidak ada LR yang stabil, diuji 1e-5. Perluasan dilakukan satu kali. Kalau LR baru kembali menjadi yang terbaik, LR itu dipakai dan dicatat sebagai "masih di tepi setelah perluasan" (tidak diperluas lagi). Pemilihan ulang memakai semua run yang stabil dan menulis `lr_selection.csv` untuk naskah |
| Semua sesi `full` harus memakai versi pustaka yang sama | Versi dikunci sesuai `environment.txt` sesi `lr_search`: grad-cam 1.5.7, captum 0.9.0, codecarbon 3.3.1, timm 1.0.26. torch 2.10.0 dan torchvision 0.25.0 berasal dari image Kaggle, sehingga notebook memberi peringatan bila berbeda. Di **Settings → Environment**, pilih **Pin to original environment** |
| Protokol penyelamatan memakai grid LR | Grid penyelamatan kini memakai gabungan grid awal dan perluasan (1e-5 sampai 1e-3) |

## Perubahan v3.2 (setelah pilot v3.1 dan revisi posisi kebaruan)
| Kebutuhan | Perubahan v3.2 |
|---|---|
| Hasil harus bisa dibandingkan langsung dengan Cedro & Chlebus (2026) | Metrik **Relevance Rank Accuracy (RRA)** dihitung per citra di `xai_utils.py`. Untuk run lama (pilot v3.1), RRA diturunkan dari `iou_topk` di `analysis.py` dengan rumus eksak RRA = 2·IoU/(1+IoU), sehingga pilot tidak perlu diulang |
| RQ6 (baru): apakah efek ukuran model bergantung pada metode XAI? | `analysis.py` menghitung selisih-dari-selisih per citra, misalnya (L − S)<sub>Grad-CAM</sub> − (L − S)<sub>IG</sub>, dengan bootstrap, Wilcoxon, dan Holm. Kontras yang ditetapkan: Grad-CAM vs IG pada `energy_norm`; pasangan lain sebagai sensitivitas (`rq6_method_dependence.csv`) |
| Tabel interpretasi 2.2 di dokumen desain harus diterapkan secara konsisten | `analysis.py` membuat `rq5_rq6_reading.csv`: per dataset dan pasangan varian, arah lokalisasi Grad-CAM, ketergantungan lesi, deletion, IG, dan kontras RQ6, beserta bacaan menurut tabel 2.2 |
| Sel analisis hanya mencetak 20.000 karakter terakhir, sehingga tabel utama tidak terlihat | Laporan dicetak utuh |

## Perubahan v3.1 (review v3, sebelum pilot)
| Masalah di v3 | Perbaikan v3.1 |
|---|---|
| Pemilihan LR (sel 6) ikut menghitung run yang divergen/kolaps, sehingga LR yang tidak stabil bisa terpilih | Hanya run `completed`/`early_stop` yang dipertimbangkan; kombinasi yang gagal di semua LR diberi peringatan dan tidak diberi LR |
| Batas `--max_bad_loss 0.2` membiarkan satu varian melewati sampai 20% batch tanpa terlihat | Batas **0,02**; total fraksi batch yang dilewati disimpan di `summary.json` dan dilaporkan per varian |
| Run gagal langsung dibuang: bias bila yang paling sering gagal adalah L | **Protokol penyelamatan pra-spesifikasi** (plan `full`, `native_res`, `datafrac`): percobaan 1 = seed dan split sama dengan LR satu tingkat lebih rendah; percobaan 2 = LR itu dalam FP32. Percobaan gagal diarsipkan (`__failed-attempt<k>`), tabel `training_stability.csv` melaporkan jumlah run berhasil, diselamatkan, dan gagal per varian, dan analisis sensitivitas tanpa run yang diselamatkan ditulis ke `summary_no_rescued/` |
| Hasil `lr_search` (`results_lr`) tidak ikut dipulihkan, jadi sesi yang terpotong harus diulang dari awal | Disimpan di `results_lr_v3/` dan dipulihkan dari input seperti `results_v3/` |
| Pilot memakai `tag="main"`, sehingga run pilot (15/5 epoch) berbagi folder dengan run `full` (BUSI fold 0, HAM10000 seed 1). Kalau output pilot dipulihkan, `full` menganggap run itu sudah selesai dan melewatinya | Pilot memakai `tag="pilot"` dan folder sendiri (`results_v3/pilot/`); `analysis.py` melaporkannya terpisah dari `main` |
| `oracle_grid` disebut batas atas skor | Dikoreksi: batas atas hanya untuk `hit`/`iou_topk`, bukan untuk `energy` (pilot: Grad-CAM++ 0,83 > oracle 0,78 di HAM10000) |

## Perubahan dari v2 (hasil pilot)
Pada pilot v2 (LR 3e-4) varian **L gagal di kedua dataset**: HAM10000 loss menjadi NaN mulai epoch 1, BUSI kolaps menebak satu kelas (loss tetap ln 3). S normal (macro-F1 0.78 BUSI, 0.69 HAM10000). Perbaikan v3, berlaku sama untuk semua varian:

| Masalah | Perbaikan v3 |
|---|---|
| LR 3e-4 terlalu besar untuk L | `DEFAULT_LR = 1e-4`; grid `lr_search` digeser ke `[3e-5, 1e-4, 3e-4]` |
| Warm-up hanya 1 epoch (BUSI ≈ 9 langkah) | warm-up = max(1 epoch, 100 langkah), maksimal 1/4 training |
| Head acak memberi loss awal 3–4 dan gradien besar ke backbone | classifier diinisialisasi nol (loss awal = ln K) |
| NaN tidak terdeteksi, run tetap lanjut dan ikut dianalisis | loss non-finite dilewati; bila > 2% (v3: 20%) batch satu epoch → status `diverged@ep`; model yang menebak satu kelas → `collapsed`. Keduanya tanpa XAI dan dikeluarkan dari `analysis.py` (dilaporkan di bagian *Excluded runs*) |
| Tidak ada opsi bila FP16 tetap overflow | flag `--no_amp` (FP32) lewat `EXTRA_TRAIN_ARGS` |
| Hasil pilot v2 bisa ikut terpulihkan | hasil disimpan di `results_v3/`; hanya folder `results_v3` yang dipulihkan dari input |

Plan `pilot` kini memuat **S, M, dan L** (v2 hanya S dan L untuk menghemat kuota), supaya stabilitas M juga dicek sebelum `lr_search`. Versi paket dicatat di `environment.txt`.

`history.csv` kini juga mencatat `bad_loss_frac` dan `val_n_pred_classes` per epoch, dan sel 5 menampilkan tabel status semua run.

## Perubahan dari v1 (menjawab hasil review)
| Kode | Masalah di v1 | Perbaikan di v2 |
|---|---|---|
| M1 | Bobot S/M (resep torchvision) dan L (port Google) berbeda asal; L dinormalisasi dengan mean/std yang salah | Bobot **timm `tf_efficientnetv2_{s,m,l}.in1k`** (ketiganya dari checkpoint resmi Google, resep sama), mean/std dari `pretrained_cfg` tiap model |
| M2 | Statistik BatchNorm dihitung dari micro-batch 32/16/8 | **BN dibekukan** (`BN_MODE="frozen"`): aturan update identik untuk semua varian, micro-batch tidak lagi berpengaruh |
| M3 | Lokalisasi tanpa pembanding; model vs metode XAI tercampur | Peta pembanding `base_center`, `base_random`, `oracle_grid`; skor `energy_norm` ternormalisasi peluang; **uji ketergantungan lesi** tanpa XAI; analisis pada citra yang benar untuk semua varian |
| M4 | Batas resolusi CAM 12×12 tidak diukur | `oracle_grid` = skor CAM yang mengikuti masker persis pada resolusi ini (batas atas untuk `hit`/`iou_topk`, bukan untuk `energy`); masker yang hilang setelah resize dicatat |
| M5 | H5 (ERF) dikorelasikan dengan n = 3 | ERF dilaporkan deskriptif saja |
| M6 | Wilcoxon n = 5 tidak mungkin signifikan; tanpa endpoint primer | `analysis.py`: bootstrap berpasangan per citra, TOST dengan margin pra-registrasi, Holm per keluarga, LMM + LRT untuk H4, satu endpoint primer |
| M7 | FLOPs 6× salah untuk 384 px; memori/throughput training tidak sebanding | `bench.py` melaporkan GMAC dan **langkah training terkontrol (batch 8)**; CodeCarbon `tracking_mode="process"` |
| minor | LR BUSI dipilih di fold 0 (bocor ke test fold lain), flip pada rontgen, sanity hanya 2 blok, insertion/deletion tanpa pembanding, IG dihaluskan diam-diam, native res tidak ada | LR search **per fold** (nested), tanpa flip untuk SIIM, sanity **cascade 6 tingkat**, AUC acak per citra, `ig` dan `ig_raw` dilaporkan terpisah, plan `native_res` |

## Sebelum menjalankan
1. **Settings → Accelerator: GPU T4 x2**, dan **Internet: On** (untuk `pip install` dan unduh bobot dari Hugging Face; akun perlu verifikasi nomor telepon).
2. **Add Input**:
   - `kmader/skin-cancer-mnist-ham10000` (citra dan metadata HAM10000)
   - `tschandl/ham10000-lesion-segmentations` (masker HAM10000)
   - `vbookshelf/pneumothorax-chest-xray-images-and-masks` (SIIM-ACR)
   - **Curated BUSI**: unduh zip dari https://zenodo.org/records/19047974, lalu *Datasets → New Dataset → upload*, lalu tambahkan sebagai input.
3. Urutan `MODE`: `"pilot"` → `"lr_search"` → `"lr_extend"` → `"full"` (beberapa sesi) → `"native_res"` (opsional tetapi disarankan).
4. Run panjang: **Save Version → Save & Run All (Commit)** (maksimal 12 jam per sesi).
5. **Melanjutkan ke sesi berikutnya**: jadikan output versi sebelumnya sebagai input. Folder `results/` dan `best_lr.json` otomatis disalin, dan run yang sudah punya file `DONE` dilewati.
6. **Sebelum `full`**: kunci margin ekuivalensi dan endpoint primer di `analysis.py` (`MARGINS`, `PRIMARY_XAI`) dan daftarkan di OSF. Jangan ubah setelah melihat hasil.

Semua hasil ada di `/kaggle/working/results_v3/` dan diringkas di `/kaggle/working/summary/`.

## Dasar metodologi dan sitasi
Setiap keputusan desain di notebook ini beserta rujukannya (nomor = daftar pustaka di bawah). Keputusan tanpa rujukan ditandai *aturan teknis pra-spesifikasi* dan harus ditulis apa adanya di bagian Metode naskah.

| Keputusan | Di mana | Dasar |
|---|---|---|
| Keluarga EfficientNetV2 S/M/L, checkpoint resmi Google `tf_efficientnetv2_*.in1k` lewat timm | `train.py` `WEIGHTS` | [1], [2] |
| Resolusi sama 384 px untuk semua varian; resolusi bawaan sebagai analisis sensitivitas | config `SAME` / `NATIVE` | [1], [3] |
| Fine-tuning penuh dari bobot ImageNet pada citra medis | `train.py` | [4], [5] |
| LR dipilih per varian dengan grid yang sama (perbandingan antarmodel yang adil) | plan `lr_search` | [5], [6] |
| LR BUSI dipilih per fold di dalam data latih fold itu (nested, tanpa kebocoran) | plan `lr_search` | [7], [8] |
| AdamW + cosine decay | `train.py` | [9], [10] |
| Warm-up LR linear (v3: minimal 100 langkah) | `train.py` | [11] |
| Head classifier diinisialisasi nol agar gradien awal ke backbone kecil | `build_model` | dimotivasi oleh [11], [12]; bentuk nol-init adalah pilihan kami |
| FP16 dengan dynamic loss scaling; opsi FP32 bila overflow | `GradScaler`, `--no_amp` | [13] |
| Gradient clipping (norma 5) | `train.py` | [14] |
| Statistik BN dibekukan karena micro-batch kecil (8 untuk L) | `bn_mode=frozen` | [15], [16] |
| Cross-entropy berbobot invers frekuensi kelas | `train.py` | pilihan standar; Cui et al. [17] dirujuk sebagai pembanding (mereka mengusulkan bobot *effective number* sebagai pengganti invers frekuensi) |
| Run divergen (>2% loss non-finite per epoch) atau kolaps (satu kelas) diselamatkan dengan protokol tetap (LR satu tingkat lebih rendah, lalu FP32); yang tetap gagal dikeluarkan; semuanya dilaporkan per varian, plus analisis sensitivitas tanpa run yang diselamatkan | `train.py`, sel 5, `analysis.py` | *aturan teknis pra-spesifikasi*; pelaporan eksklusi mengikuti [18], [19] |
| HAM10000, split per `lesion_id`, masker segmentasi | `data_utils.py` | [20], [21], [22] |
| BUSI | `data_utils.py` | [23] + sitasi rilis Curated BUSI (Zenodo 19047974) |
| SIIM-ACR Pneumothorax | `data_utils.py` | [24], sumber citra [25] |
| Shortcut yang diketahui (chest drain, penggaris/tinta) sebagai alasan RQ5 | desain | [26], [27] |
| Grad-CAM, Grad-CAM++, Integrated Gradients | `xai_utils.py` | [28], [29], [30]; implementasi [31], [32] |
| Energi saliency di dalam masker (energy-based pointing game) dan hit (pointing game) | `xai_utils.py` | [33], [34] |
| Relevance Rank Accuracy, agar sebanding dengan karya terdekat | `xai_utils.py`, `analysis.py` | [55], dipakai oleh [56] |
| Selisih-dari-selisih antarmetode (RQ6) dan tabel interpretasi pra-spesifikasi | `analysis.py` | *aturan analisis pra-spesifikasi* (dokumen desain 2.2 dan 6.5); motivasi dari [56], [57], [58] |
| Evaluasi saliency terhadap masker/anotasi pada citra medis | desain | [35], [36] |
| Insertion/deletion dengan pembanding urutan piksel acak | `xai_utils.py` | [37], [38] |
| Uji ketergantungan lesi dengan oklusi | `xai_utils.py` | [39] |
| Sanity check randomisasi parameter bertingkat | `xai_utils.py` | [40], [41] |
| Effective receptive field (deskriptif) | `xai_utils.py` | [42] |
| Bootstrap berpasangan per citra, CI 95% | `analysis.py` | [43] |
| Uji ekuivalensi (TOST, CI 90%) dengan margin pra-registrasi | `analysis.py` | [44], [45] |
| Koreksi Holm per keluarga uji | `analysis.py` | [46] |
| Wilcoxon signed-rank sebagai pendamping | `analysis.py` | [47] |
| LMM + likelihood-ratio test untuk interaksi varian × ukuran lesi (H4) | `analysis.py` | [48], [49] |
| Tidak memakai uji per fold dengan n kecil | `analysis.py` | [50] |
| Beberapa seed/fold per varian untuk variansi training | plan `full` | [51] |
| Efisiensi: GMAC, latensi, throughput dilaporkan bersama | `bench.py` | [1], [52] |
| Estimasi energi dengan CodeCarbon | `train.py` | [53], [54] |
| Checklist pelaporan naskah | – | CLAIM 2024 [18], TRIPOD+AI [19] |

### Daftar pustaka
1. Tan, M., & Le, Q. V. (2021). EfficientNetV2: Smaller models and faster training. *ICML*. arXiv:2104.00298
2. Wightman, R. (2019). PyTorch Image Models (timm). GitHub. doi:10.5281/zenodo.4414861
3. Touvron, H., Vedaldi, A., Douze, M., & Jégou, H. (2019). Fixing the train-test resolution discrepancy. *NeurIPS*. arXiv:1906.06423
4. Raghu, M., Zhang, C., Kleinberg, J., & Bengio, S. (2019). Transfusion: Understanding transfer learning for medical imaging. *NeurIPS*. arXiv:1902.07208
5. Kornblith, S., Shlens, J., & Le, Q. V. (2019). Do better ImageNet models transfer better? *CVPR*. arXiv:1805.08974
6. Dodge, J., Gururangan, S., Card, D., Schwartz, R., & Smith, N. A. (2019). Show your work: Improved reporting of experimental results. *EMNLP*. arXiv:1909.03004
7. Varma, S., & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics*, 7, 91. doi:10.1186/1471-2105-7-91
8. Varoquaux, G., & Cheplygina, V. (2022). Machine learning for medical imaging: methodological failures and recommendations for the future. *npj Digital Medicine*, 5, 48. doi:10.1038/s41746-022-00592-y
9. Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*. arXiv:1711.05101
10. Loshchilov, I., & Hutter, F. (2017). SGDR: Stochastic gradient descent with warm restarts. *ICLR*. arXiv:1608.03983
11. Goyal, P., et al. (2017). Accurate, large minibatch SGD: Training ImageNet in 1 hour. arXiv:1706.02677
12. Kumar, A., Raghunathan, A., Jones, R., Ma, T., & Liang, P. (2022). Fine-tuning can distort pretrained features and underperform out-of-distribution. *ICLR*. arXiv:2202.10054
13. Micikevicius, P., et al. (2018). Mixed precision training. *ICLR*. arXiv:1710.03740
14. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). On the difficulty of training recurrent neural networks. *ICML*. arXiv:1211.5063
15. Ioffe, S., & Szegedy, C. (2015). Batch normalization. *ICML*. arXiv:1502.03167
16. Wu, Y., & He, K. (2018). Group normalization. *ECCV*. arXiv:1803.08494
17. Cui, Y., Jia, M., Lin, T.-Y., Song, Y., & Belongie, S. (2019). Class-balanced loss based on effective number of samples. *CVPR*. arXiv:1901.05555
18. Tejani, A. S., et al. (2024). Checklist for Artificial Intelligence in Medical Imaging (CLAIM): 2024 update. *Radiology: Artificial Intelligence*, 6(4), e240300. doi:10.1148/ryai.240300
19. Collins, G. S., et al. (2024). TRIPOD+AI statement. *BMJ*, 385, e078378. doi:10.1136/bmj-2023-078378
20. Tschandl, P., Rosendahl, C., & Kittler, H. (2018). The HAM10000 dataset. *Scientific Data*, 5, 180161. doi:10.1038/sdata.2018.161
21. Tschandl, P., et al. (2020). Human–computer collaboration for skin cancer recognition. *Nature Medicine*, 26, 1229–1234. doi:10.1038/s41591-020-0942-0 (sumber masker HAM10000; cocokkan dengan halaman dataset Kaggle)
22. Codella, N., et al. (2019). Skin lesion analysis toward melanoma detection 2018 (ISIC 2018). arXiv:1902.03368
23. Al-Dhabyani, W., Gomaa, M., Khaled, H., & Fahmy, A. (2020). Dataset of breast ultrasound images. *Data in Brief*, 28, 104863. doi:10.1016/j.dib.2019.104863
24. SIIM-ACR Pneumothorax Segmentation challenge (2019). Kaggle/SIIM. Sitasi sesuai syarat kompetisi.
25. Wang, X., et al. (2017). ChestX-ray8: Hospital-scale chest X-ray database and benchmarks. *CVPR*. arXiv:1705.02315
26. Oakden-Rayner, L., Dunnmon, J., Carneiro, G., & Ré, C. (2020). Hidden stratification causes clinically meaningful failures in machine learning for medical imaging. *ACM CHIL*. arXiv:1909.12475
27. Bissoto, A., Fornaciali, M., Valle, E., & Avila, S. (2019). (De)Constructing bias on skin lesion datasets. *CVPR Workshops*. arXiv:1904.08818
28. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV*. doi:10.1109/ICCV.2017.74
29. Chattopadhay, A., Sarkar, A., Howlader, P., & Balasubramanian, V. N. (2018). Grad-CAM++. *WACV*. arXiv:1710.11063
30. Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic attribution for deep networks. *ICML*. arXiv:1703.01365
31. Gildenblat, J., et al. (2021). pytorch-grad-cam. GitHub.
32. Kokhlikyan, N., et al. (2020). Captum: A unified and generic model interpretability library for PyTorch. arXiv:2009.07896
33. Wang, H., et al. (2020). Score-CAM: Score-weighted visual explanations for convolutional neural networks. *CVPR Workshops*. arXiv:1910.01279
34. Zhang, J., Bargal, S. A., Lin, Z., Brandt, J., Shen, X., & Sclaroff, S. (2018). Top-down neural attention by excitation backprop. *IJCV*, 126, 1084–1102. arXiv:1608.00507
35. Arun, N., et al. (2021). Assessing the trustworthiness of saliency maps for localizing abnormalities in medical imaging. *Radiology: Artificial Intelligence*, 3(6), e200267. doi:10.1148/ryai.2021200267
36. Saporta, A., et al. (2022). Benchmarking saliency methods for chest X-ray interpretation. *Nature Machine Intelligence*, 4, 867–878. doi:10.1038/s42256-022-00536-x
37. Petsiuk, V., Das, A., & Saenko, K. (2018). RISE: Randomized input sampling for explanation of black-box models. *BMVC*. arXiv:1806.07421
38. Hooker, S., Erhan, D., Kindermans, P.-J., & Kim, B. (2019). A benchmark for interpretability methods in deep neural networks. *NeurIPS*. arXiv:1806.10758
39. Zeiler, M. D., & Fergus, R. (2014). Visualizing and understanding convolutional networks. *ECCV*. arXiv:1311.2901
40. Adebayo, J., Gilmer, J., Muelly, M., Goodfellow, I., Hardt, M., & Kim, B. (2018). Sanity checks for saliency maps. *NeurIPS*. arXiv:1810.03292
41. Tomsett, R., Harborne, D., Chakraborty, S., Gurram, P., & Preece, A. (2020). Sanity checks for saliency metrics. *AAAI*. arXiv:1912.01451
42. Luo, W., Li, Y., Urtasun, R., & Zemel, R. (2016). Understanding the effective receptive field in deep convolutional neural networks. *NeurIPS*. arXiv:1701.04128
43. Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
44. Schuirmann, D. J. (1987). A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability. *Journal of Pharmacokinetics and Biopharmaceutics*, 15, 657–680. doi:10.1007/BF01068419
45. Lakens, D. (2017). Equivalence tests: A practical primer. *Social Psychological and Personality Science*, 8(4), 355–362. doi:10.1177/1948550617697177
46. Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70.
47. Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin*, 1(6), 80–83. doi:10.2307/3001968
48. Bates, D., Mächler, M., Bolker, B., & Walker, S. (2015). Fitting linear mixed-effects models using lme4. *Journal of Statistical Software*, 67(1). doi:10.18637/jss.v067.i01
49. Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proc. 9th Python in Science Conference*.
50. Nadeau, C., & Bengio, Y. (2003). Inference for the generalization error. *Machine Learning*, 52, 239–281. doi:10.1023/A:1024068626366
51. Bouthillier, X., et al. (2021). Accounting for variance in machine learning benchmarks. *MLSys*. arXiv:2103.03098
52. Dehghani, M., Arnab, A., Beyer, L., Vaswani, A., & Tay, Y. (2022). The efficiency misnomer. *ICLR*. arXiv:2110.12894
53. Courty, B., et al. CodeCarbon (software), Zenodo. Sitasi versi yang tercatat di `environment.txt`.
54. Henderson, P., et al. (2020). Towards the systematic reporting of the energy and carbon footprints of machine learning. *JMLR*, 21(248), 1–43. arXiv:2002.05651
55. Arras, L., Osman, A., & Samek, W. (2022). CLEVR-XAI: A benchmark dataset for the ground truth evaluation of neural network explanations. *Information Fusion*, 81. doi:10.1016/j.inffus.2021.11.008
56. Cedro, M., & Chlebus, M. (2026). Scaling vision models does not consistently improve localisation-based explanation quality. arXiv:2605.10142 (preprint)
57. Santos, R., Pedrosa, J., Mendonça, A. M., & Campilho, A. (2025). Grad-CAM: The impact of large receptive fields and other caveats. *Computer Vision and Image Understanding*, 258. doi:10.1016/j.cviu.2025.104383
58. Yasuki, S., & Taki, M. (2024). CAM Back Again: Large kernel CNNs from a weakly supervised object localization perspective. *CVPR*, 341–351. arXiv:2403.06676

**Catatan verifikasi:** penulis utama, judul, venue, dan tahun adalah rujukan standar, tetapi nomor DOI/arXiv, volume, dan halaman ditulis dari ingatan. Cek setiap entri di Google Scholar atau Crossref sebelum masuk naskah, terutama [2], [21], [24], [31], [53]. Karya terdekat untuk bagian Related Work (Santos et al., 2025; Zhang et al., Radiology: AI, doi 10.1148/ryai.220221; "Sharpness matters", Sci. Rep. 2026) ada di dokumen desain.

In [ ]:
# Versi dikunci (v3.3) sesuai environment.txt sesi lr_search, supaya semua sesi memakai pustaka yang sama.
!pip install -q "grad-cam==1.5.7" "captum==0.9.0" "codecarbon==3.3.1" "timm==1.0.26"
import torch, torchvision, timm
print(torch.__version__, torchvision.__version__, "timm", timm.__version__, "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
EXPECTED = {"torch": "2.10.0", "torchvision": "0.25.0", "timm": "1.0.26"}
_found = {"torch": torch.__version__, "torchvision": torchvision.__version__, "timm": timm.__version__}
_diff = {k: (_found[k], v) for k, v in EXPECTED.items() if not _found[k].startswith(v)}
if _diff:
    print("PERINGATAN: versi berbeda dari sesi lr_search (ditemukan, diharapkan):", _diff,
          "-> di Settings pilih 'Pin to original environment', atau catat perbedaan ini di naskah.")
# catat versi paket untuk bagian "Implementation details" naskah (reprodusibilitas)
import subprocess, sys
open("/kaggle/working/environment.txt", "w").write(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)


## 1. Konfigurasi

In [ ]:
# ============================== CONFIG ==============================
MODE = "pilot"          # "pilot" | "lr_search" | "lr_extend" | "full" | "native_res" | "datafrac"
IMG_SIZE = 384          # resolusi SAMA untuk semua varian pada eksperimen utama (peta fitur terakhir 12x12)
BN_MODE = "frozen"      # "frozen" = statistik BN ImageNet dibekukan (rekomendasi), "train" = BN biasa

DATASET_PATHS = {       # None = deteksi otomatis di /kaggle/input
    "busi": None, "ham10000_images": None, "ham10000_masks": None, "siim": None,
}

SAME = {"s": IMG_SIZE, "m": IMG_SIZE, "l": IMG_SIZE}
NATIVE = {"s": 384, "m": 480, "l": 480}   # resolusi uji bawaan checkpoint tf_efficientnetv2_*.in1k
XAI_FULL = dict(xai_n=300, insdel_n=100, reliance_n=300, sanity_n=30, scorecam_n=0)

PLANS = {
    # S, M, L, 1 ulangan, epoch pendek: cek pipeline, stabilitas training tiap varian, kalibrasi waktu.
    # Hasil pilot TIDAK dilaporkan sebagai temuan (hanya uji kelayakan), karena n = 1 per varian.
    # tag="pilot" (v3.1): dengan tag="main", run pilot (15/5 epoch) punya folder yang sama dengan run full untuk
    # BUSI fold 0 dan HAM10000 seed 1, sehingga plan full menganggapnya sudah DONE dan melewatinya.
    "pilot": dict(datasets=["busi", "ham10000"], variants=["s", "m", "l"], busi_folds=[0], seeds=[1],
                  epochs={"busi": 15, "ham10000": 5}, sizes=SAME, tag="pilot",   # v3.1: folder sendiri, lihat catatan
                  xai_n=50, insdel_n=30, reliance_n=50, sanity_n=10, scorecam_n=0),
    # pencarian learning rate, tanpa XAI. BUSI: per fold (nested) supaya LR tidak dipilih memakai citra test fold lain
    "lr_search": dict(datasets=["busi", "ham10000", "siim"], variants=["s", "m", "l"],
                      busi_folds=[0, 1, 2, 3, 4], seeds=[1], lrs=[3e-5, 1e-4, 3e-4], epochs=10, sizes=SAME),
    # v3.3: perluasan grid satu kali untuk kombinasi yang LR terbaiknya di tepi grid lr_search (butuh results_lr_v3 sebagai input)
    "lr_extend": dict(datasets=["busi", "ham10000", "siim"], variants=["s", "m", "l"],
                      busi_folds=[0, 1, 2, 3, 4], seeds=[1], ext_low=1e-5, ext_high=1e-3, epochs=10, sizes=SAME),
    # eksperimen utama (dijalankan bertahap di beberapa sesi)
    "full": dict(datasets=["busi", "ham10000", "siim"], variants=["s", "m", "l"],
                 busi_folds=[0, 1, 2, 3, 4], seeds=[1, 2, 3, 4, 5],
                 epochs={"busi": 50, "ham10000": 30, "siim": 30}, sizes=SAME, tag="main", **XAI_FULL),
    # analisis sensitivitas: tiap varian pada resolusi bawaannya (BUSI murah; ganti ke ham10000 bila kuota cukup)
    "native_res": dict(datasets=["busi"], variants=["s", "m", "l"], busi_folds=[0, 1, 2, 3, 4], seeds=[1],
                       epochs={"busi": 50}, sizes=NATIVE, tag="native", **XAI_FULL),
    # future work (bukan bagian inti skripsi): fraksi data latih
    "datafrac": dict(datasets=["ham10000"], variants=["s", "m", "l"], busi_folds=[0], seeds=[1, 2, 3],
                     fracs=[0.25, 0.5], epochs={"ham10000": 30}, sizes=SAME, tag="datafrac",
                     xai_n=300, insdel_n=100, reliance_n=300, sanity_n=0, scorecam_n=0),
}
DEFAULT_LR = 1e-4          # v3: 3e-4 membuat L divergen di pilot v2; dipakai sama untuk semua varian sampai ada best_lr.json
PATIENCE = 7
SIIM_NEG_PER_POS = 1.0          # 1.0 = seimbang (2.669 positif + 2.669 negatif acak, seed tetap)
STOP_LAUNCH_AFTER_H = 9.0       # jangan mulai run baru setelah jam ke-9 sesi
HARD_DEADLINE_H = 11.3          # run yang masih berjalan dihentikan dan diulang di sesi berikutnya
KEEP_CKPT = False               # True = simpan bobot terbaik (FP16) tiap run
WORKERS_PER_RUN = 2             # DataLoader workers per proses (4 vCPU dibagi 2 proses)
EXTRA_TRAIN_ARGS = []           # mis. ["--max_steps", "5"] untuk uji cepat, ["--grad_ckpt"] bila L kehabisan memori.
                                # Jangan menaruh "--no_amp" di sini untuk plan full: FP32 sudah menjadi langkah 2 protokol penyelamatan.

# Protokol penyelamatan run gagal (diverged/collapsed), ditetapkan SEBELUM eksperimen dan sama untuk semua varian:
#   percobaan 1: seed dan split sama, LR satu tingkat lebih rendah di grid lr_search (atau LR/3 bila sudah terendah)
#   percobaan 2: LR percobaan 1, dilatih dalam FP32
# Tidak dipakai di pilot (kegagalan harus terlihat mentah) dan lr_search (kegagalan adalah informasi pemilihan LR).
RESCUE_MODES = ("full", "native_res", "datafrac")

# ---- Pembagian plan "full" ke beberapa akun (v3.4) ----
# Setiap akun HARUS milik orang yang berbeda dan dijalankan oleh pemiliknya sendiri (Terms of Use Kaggle:
# satu orang hanya boleh memakai satu akun; akun dan password tidak boleh dipinjamkan).
# SHARD = None  -> semua run (satu akun) dan sesi penggabungan akhir (benchmark + analisis gabungan)
# SHARD = 1/2/3 -> hanya run milik akun itu. Isi SHARDS jangan diubah setelah pembagian dimulai.
SHARD = None
SHARDS = {
    1: {"ham10000": [1, 2, 3]},                                  # sekitar 21 jam GPU
    2: {"ham10000": [4, 5], "busi": [0, 1, 2, 3, 4]},            # sekitar 20 jam GPU
    3: {"siim": [1, 2, 3, 4, 5]},                                # sekitar 23 jam GPU
}
MAX_RESCUE = 2

INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
RESULTS = f"{WORK}/results_v3"   # folder baru: hasil pilot v2 (L rusak) tidak ikut dipulihkan/dianalisis
RESULTS_LR = f"{WORK}/results_lr_v3"   # hasil lr_search, ikut dipulihkan antar sesi
CACHE = "/tmp/cache"            # tidak ikut tersimpan ke output

## 2. Modul kode (ditulis ke /kaggle/working)

### Modul `data_utils.py`: indeks dataset, cache per resolusi, split tanpa kebocoran

In [ ]:
%%writefile data_utils.py
"""Dataset indexing, caching (pre-resize), and splitting for the EfficientNetV2 S/M/L x XAI study.

Layout under <cache_root>/<dataset>/:
    index.csv            uid, image_src, mask_src, label, label_name, group   (size independent)
    splits.json          run_id -> {train, val, test} lists of uid        (size independent)
    <size>/img/<uid>.jpg
    <size>/mask/<uid>.png
    <size>/masks.csv     uid, mask_area, mask_lost  (mask_lost = lesion vanished after resizing)

Splits do not depend on the image size, so the 384 px and native-resolution runs use the same test images.
"""
import glob
import json
import os
import re
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, train_test_split

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


def _find(root, pattern):
    return sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))


def _fail(name, root, hint):
    tree = []
    for dirpath, dirnames, filenames in os.walk(root or "."):
        depth = dirpath[len(root or "."):].count(os.sep)
        if depth > 3:
            continue
        tree.append(f"{dirpath}  ({len(filenames)} files)")
        if len(tree) > 40:
            break
    raise RuntimeError(
        f"[{name}] could not index dataset under {root}.\n{hint}\n"
        "Folder tree (first levels):\n  " + "\n  ".join(tree)
        + "\nSet DATASET_PATHS in the CONFIG cell to the correct folder."
    )


# ---------------------------------------------------------------- BUSI (curated)
BUSI_CLASSES = ["normal", "benign", "malignant"]


def index_busi(root):
    """Works for the original BUSI layout (<class>/<name>.png + <name>_mask.png, extra masks <name>_mask_1.png)
    and for re-packaged versions that keep class folders or class prefixes in file names.
    Multiple masks for one image are merged (union). BUSI has no patient ID, so group = image
    (limitation reported in the paper)."""
    if not root:
        _fail("busi", root, "Path not set.")
    files = [f for f in _find(root, "*") if f.lower().endswith(IMG_EXT)]
    if not files:
        _fail("busi", root, "No image files found.")
    masks, masks_by_stem, images = {}, {}, []
    for f in files:
        base = os.path.splitext(os.path.basename(f))[0]
        m = re.match(r"^(.*?)[ _-]?(mask|Mask|MASK)(?:[ _-]?\d+)?$", base)
        if m:
            stem = m.group(1).rstrip(" _-")
            masks.setdefault((os.path.dirname(f), stem), []).append(f)
            masks_by_stem.setdefault(stem, []).append(f)
        else:
            images.append(f)
    rows = []
    for f in images:
        base = os.path.splitext(os.path.basename(f))[0]
        low = f.lower()
        cls = None
        for c in BUSI_CLASSES:
            if f"{os.sep}{c}{os.sep}" in low or os.path.basename(low).startswith(c):
                cls = c
                break
        if cls is None:
            continue
        mk = masks.get((os.path.dirname(f), base), []) or masks_by_stem.get(base, [])
        uid = f"busi_{len(rows):05d}"
        rows.append(dict(uid=uid, image_src=f, mask_src="|".join(sorted(mk)),
                         label=BUSI_CLASSES.index(cls), label_name=cls, group=uid))
    df = pd.DataFrame(rows)
    if df.empty or df.label.nunique() < 3:
        _fail("busi", root, "Expected class folders or file prefixes normal/benign/malignant.")
    return df


# ---------------------------------------------------------------- HAM10000
HAM_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]


def index_ham(root_images, root_masks):
    metas = [p for p in _find(root_images, "*.csv") if "metadata" in os.path.basename(p).lower()]
    if not metas:
        _fail("ham10000", root_images, "HAM10000_metadata.csv (with lesion_id, image_id, dx) not found.")
    meta = pd.read_csv(metas[0])
    imgs = {}
    for p in _find(root_images, "ISIC_*.jpg") + _find(root_images, "ISIC_*.png"):
        if "segmentation" in os.path.basename(p).lower():
            continue
        imgs.setdefault(re.search(r"ISIC_\d+", p).group(0), p)
    msk = {}
    for p in _find(root_masks, "ISIC_*"):
        if p.lower().endswith(IMG_EXT):
            msk.setdefault(re.search(r"ISIC_\d+", p).group(0), p)
    rows = []
    for r in meta.itertuples():
        if r.image_id not in imgs:
            continue
        rows.append(dict(uid=r.image_id, image_src=imgs[r.image_id], mask_src=msk.get(r.image_id, ""),
                         label=HAM_CLASSES.index(r.dx), label_name=r.dx, group=r.lesion_id))
    df = pd.DataFrame(rows)
    if df.empty:
        _fail("ham10000", root_images, "No ISIC_*.jpg images matched the metadata.")
    print(f"[ham10000] {len(df)} images, {int((df['mask_src'] != '').sum())} with masks")
    return df


# ---------------------------------------------------------------- SIIM-ACR (PNG version)
def index_siim(root, n_neg_per_pos=1.0, seed=42):
    """Pairs every image with a mask of the same file name in a folder whose path contains 'mask'.
    Label = mask has any positive pixel. Negatives are sub-sampled with a fixed seed (class balancing)."""
    if not root:
        _fail("siim", root, "Path not set.")
    files = _find(root, "*.png")
    is_mask = lambda f: "mask" in os.path.relpath(f, root).lower()  # the Kaggle slug itself contains "masks"
    mask_files = {os.path.basename(f): f for f in files if is_mask(f)}
    img_files = [f for f in files if not is_mask(f)]
    if not img_files or not mask_files:
        _fail("siim", root, "Expected separate image and mask folders with matching PNG file names.")
    rows = []
    for f in img_files:
        m = mask_files.get(os.path.basename(f))
        if m is not None:
            rows.append(dict(uid=os.path.splitext(os.path.basename(f))[0], image_src=f, mask_src=m))
    df = pd.DataFrame(rows)
    with ProcessPoolExecutor(4) as ex:
        pos = list(ex.map(_mask_positive, df["mask_src"], chunksize=64))
    df["label"] = np.array(pos, dtype=int)
    df["label_name"] = np.where(df.label == 1, "pneumothorax", "normal")
    df.loc[df.label == 0, "mask_src"] = ""
    n_pos = int(df.label.sum())
    neg = df[df.label == 0].sample(n=min(int(n_pos * n_neg_per_pos), int((df.label == 0).sum())),
                                   random_state=seed)
    df = pd.concat([df[df.label == 1], neg]).sort_values("uid").reset_index(drop=True)
    df["group"] = df["uid"]
    print(f"[siim] {n_pos} positive + {len(neg)} sampled negative")
    return df


def _mask_positive(p):
    return int(np.asarray(Image.open(p).convert("L")).max() > 0)


# ---------------------------------------------------------------- caching
def _cache_one(args):
    src, msrc, dst, mdst, size = args
    if not os.path.exists(dst):
        Image.open(src).convert("RGB").resize((size, size), Image.BICUBIC).save(dst, quality=95)
    area, lost = np.nan, False
    if msrc:
        orig_pos = False
        m = None
        for p in msrc.split("|"):
            im = Image.open(p).convert("L")
            orig_pos = orig_pos or (np.asarray(im).max() > 127)
            a = np.asarray(im.resize((size, size), Image.NEAREST)) > 127
            m = a if m is None else (m | a)
        area = float(m.mean())
        lost = bool(orig_pos and area == 0)
        if not os.path.exists(mdst):
            Image.fromarray((m * 255).astype(np.uint8)).save(mdst)
    return area, lost


def build_cache(df, ds_dir, size, workers=4):
    d = os.path.join(ds_dir, str(size))
    os.makedirs(os.path.join(d, "img"), exist_ok=True)
    os.makedirs(os.path.join(d, "mask"), exist_ok=True)
    jobs = [(r.image_src, r.mask_src, os.path.join(d, "img", f"{r.uid}.jpg"),
             os.path.join(d, "mask", f"{r.uid}.png"), size) for r in df.itertuples()]
    with ProcessPoolExecutor(workers) as ex:
        res = list(ex.map(_cache_one, jobs, chunksize=32))
    ms = pd.DataFrame({"uid": df.uid.values, "mask_area": [r[0] for r in res], "mask_lost": [r[1] for r in res]})
    ms.to_csv(os.path.join(d, "masks.csv"), index=False)
    n_lost = int(ms.mask_lost.sum())
    if n_lost:
        print(f"  WARNING: {n_lost} lesion masks became empty after resizing to {size}px (excluded from XAI; report this)")
    return ms


# ---------------------------------------------------------------- splits
def make_splits(df, dataset, n_folds=5, test_frac=0.15, val_frac=0.15, seed=42):
    """Returns dict run_id -> dict(train=[uid], val=[uid], test=[uid]).
    busi: 5-fold CV (run_id = fold; val is carved out of that fold's train part, so learning-rate
    selection on it never touches the fold's test images).
    ham10000/siim: one fixed split grouped by `group` (runs differ only in seed), run_id 'fixed'."""
    y, g, uid = df.label.values, df.group.values, df.uid.values
    splits = {}
    if dataset == "busi":
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
        for k, (trv, te) in enumerate(skf.split(uid, y)):
            tr, va = train_test_split(trv, test_size=val_frac / (1 - 1 / n_folds), stratify=y[trv],
                                      random_state=seed)
            splits[str(k)] = dict(train=uid[tr].tolist(), val=uid[va].tolist(), test=uid[te].tolist())
    else:
        sgkf = StratifiedGroupKFold(n_splits=int(round(1 / test_frac)), shuffle=True, random_state=seed)
        trv, te = next(sgkf.split(uid, y, g))
        sgkf2 = StratifiedGroupKFold(n_splits=int(round((1 - test_frac) / val_frac)), shuffle=True,
                                     random_state=seed)
        tr_i, va_i = next(sgkf2.split(uid[trv], y[trv], g[trv]))
        splits["fixed"] = dict(train=uid[trv][tr_i].tolist(), val=uid[trv][va_i].tolist(),
                               test=uid[te].tolist())
    # leakage guard: no group may appear in two partitions
    grp = dict(zip(uid, g))
    for rid, s in splits.items():
        gs = {k: {grp[u] for u in s[k]} for k in ("train", "val", "test")}
        assert not (gs["train"] & gs["test"]) and not (gs["val"] & gs["test"]) and not (gs["train"] & gs["val"]), \
            f"group leakage in split {rid}"
    return splits


def prepare(dataset, paths, cache_root, sizes, workers=4, siim_neg_ratio=1.0):
    """Index + split (once) and cache every requested size. Returns (index_df, splits)."""
    out_dir = os.path.join(cache_root, dataset)
    os.makedirs(out_dir, exist_ok=True)
    idx_csv, split_json = os.path.join(out_dir, "index.csv"), os.path.join(out_dir, "splits.json")
    if os.path.exists(idx_csv) and os.path.exists(split_json):
        df, splits = pd.read_csv(idx_csv, keep_default_na=False), json.load(open(split_json))
    else:
        if dataset == "busi":
            df = index_busi(paths["busi"])
        elif dataset == "ham10000":
            df = index_ham(paths["ham10000_images"], paths["ham10000_masks"])
        elif dataset == "siim":
            df = index_siim(paths["siim"], n_neg_per_pos=siim_neg_ratio)
        else:
            raise ValueError(dataset)
        splits = make_splits(df, dataset)
        df.to_csv(idx_csv, index=False)
        json.dump(splits, open(split_json, "w"))
        print(f"[{dataset}] classes: {df.label_name.value_counts().to_dict()}")
    for size in sizes:
        if not os.path.exists(os.path.join(out_dir, str(size), "masks.csv")):
            print(f"[{dataset}] caching {len(df)} images at {size}px ...")
            build_cache(df, out_dir, size, workers)
    return df, splits

### Modul `xai_utils.py`: saliency, metrik, peta pembanding, uji ketergantungan lesi, sanity bertingkat, ERF

In [ ]:
%%writefile xai_utils.py
"""Saliency methods, explanation-quality metrics, reference maps, lesion-reliance test,
cascading sanity check and effective receptive field. Model = timm EfficientNetV2
(conv_stem, bn1, blocks[stages], conv_head, bn2, global_pool, classifier)."""
import copy

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr

_trapz = getattr(np, "trapezoid", None) or np.trapz


# ---------------------------------------------------------------- saliency
def target_layer(model):
    """Last conv block output (after BN + SiLU): same 1280 x (H/32) x (W/32) map for S, M and L."""
    return model.bn2


def make_explainers(model, methods):
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
    tl = [target_layer(model)]
    ex = {}
    if "gradcam" in methods:
        ex["gradcam"] = GradCAM(model=model, target_layers=tl)
    if "gradcampp" in methods:
        ex["gradcampp"] = GradCAMPlusPlus(model=model, target_layers=tl)
    if "scorecam" in methods:
        sc = ScoreCAM(model=model, target_layers=tl)
        sc.batch_size = 32
        ex["scorecam"] = sc
    if "ig" in methods:
        from captum.attr import IntegratedGradients
        ex["ig"] = IntegratedGradients(model)
    return ex


def release(explainers):
    for e in explainers.values():
        a = getattr(e, "activations_and_grads", None)
        if a is not None:
            a.release()


def saliency(name, explainer, x, target, ig_steps=32):
    """x: [1,3,H,W] normalized tensor. Returns dict of HxW non-negative float maps.
    IG returns two maps: 'ig_raw' (|attribution| summed over channels) and 'ig' (9x9 box-smoothed,
    declared modification so pixel-level IG is comparable to CAM maps for localization)."""
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    if name == "ig":
        attr = explainer.attribute(x, baselines=torch.zeros_like(x), target=int(target),
                                   n_steps=ig_steps, internal_batch_size=4)  # 4 keeps L (FP32, 384 px) well inside 16 GB
        s = attr.detach().abs().sum(1)[0].float()
        sm = F.avg_pool2d(s[None, None], 9, stride=1, padding=4)[0, 0]
        return {"ig_raw": s.cpu().numpy(), "ig": sm.cpu().numpy()}
    cam = explainer(input_tensor=x, targets=[ClassifierOutputTarget(int(target))])
    return {name: cam[0].astype(np.float32)}


# ---------------------------------------------------------------- reference maps (no model)
def reference_maps(mask, grid, seed=0):
    """Maps that tell how good a score is:
    base_center : isotropic Gaussian at the image centre (sigma = H/4) -> centre-bias baseline
    base_random : random grid x grid map, bilinear-upsampled (same resolution as a CAM)
    oracle_grid : lesion fraction per CAM cell, bilinear-upsampled -> resolution-matched reference
                  (what a CAM of this resolution scores if it follows the mask exactly). It is close to the
                  ceiling for `hit` and `iou_topk`, but NOT for `energy`: a map concentrated on the cells fully
                  inside the lesion can exceed it (seen in the pilot: Grad-CAM++ 0.83 vs oracle 0.78 on HAM10000)."""
    H, W = mask.shape
    yy, xx = np.mgrid[0:H, 0:W]
    sig = H / 4
    center = np.exp(-(((yy - H / 2) ** 2 + (xx - W / 2) ** 2) / (2 * sig ** 2))).astype(np.float32)
    rng = np.random.default_rng(seed)
    rnd = torch.from_numpy(rng.random((grid, grid)).astype(np.float32))[None, None]
    rnd = F.interpolate(rnd, size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
    m = torch.from_numpy(mask.astype(np.float32))[None, None]
    orc = F.adaptive_avg_pool2d(m, grid)
    orc = F.interpolate(orc, size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
    return {"base_center": center, "base_random": rnd, "oracle_grid": orc}


# ---------------------------------------------------------------- metrics vs mask
def _norm(s):
    s = s - s.min()
    m = s.max()
    return s / m if m > 0 else None


def localization_metrics(sal, mask):
    """sal: HxW float, mask: HxW bool. Returns raw scores and the chance-normalized energy
    energy_norm = (energy - a) / (1 - a), a = lesion area: 0 = uniform map, 1 = all energy in lesion.
    rra = Relevance Rank Accuracy (Arras et al., 2022; used by Cedro & Chlebus, 2026): share of the
    top-k pixels (k = lesion size) that fall inside the lesion."""
    s = _norm(sal)
    area = float(mask.mean())
    if s is None:
        return dict(hit=np.nan, energy=np.nan, energy_norm=np.nan, iou50=np.nan, iou_topk=np.nan, rra=np.nan,
                    mask_area=area)
    flat = s.ravel()
    mx = flat.max()
    hit = float(mask.ravel()[flat >= mx - 1e-6].mean())       # ties at the max split the credit
    energy = float((s * mask).sum() / (s.sum() + 1e-8))
    b = s >= 0.5
    iou50 = float((b & mask).sum() / ((b | mask).sum() + 1e-8))
    k = max(1, int(mask.sum()))
    thr = np.partition(flat, -k)[-k]
    bk = s >= thr
    iou_topk = float((bk & mask).sum() / ((bk | mask).sum() + 1e-8))
    rra = float((bk & mask).sum() / max(int(bk.sum()), 1))   # ties at the threshold enlarge bk; count them all
    return dict(hit=hit, energy=energy, energy_norm=(energy - area) / max(1 - area, 1e-6),
                iou50=iou50, iou_topk=iou_topk, rra=rra, mask_area=area)


# ---------------------------------------------------------------- faithfulness
def blur(x, k=31, sigma=10.0):
    ax = torch.arange(k, device=x.device) - k // 2
    g = torch.exp(-(ax.float() ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).view(1, 1, 1, k).repeat(3, 1, 1, 1).to(x.dtype)
    y = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g, groups=3)
    return F.conv2d(F.pad(y, (0, 0, k // 2, k // 2), mode="reflect"), g.transpose(2, 3), groups=3)


@torch.no_grad()
def _probs(model, batch, target, amp, chunk=16):
    out = []
    for i in range(0, batch.shape[0], chunk):
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and batch.is_cuda):
            out.append(model(batch[i:i + chunk]).float().softmax(1)[:, int(target)])
    return torch.cat(out).cpu().numpy()


@torch.no_grad()
def insertion_deletion(model, x, target, order, steps=25, amp=True):
    """RISE-style insertion/deletion AUC (Petsiuk et al., 2018) for a given pixel order (most salient first).
    Deletion replaces pixels by 0 (= dataset mean after normalization); insertion starts from a blurred image.
    Scores must be compared with the same image's random-order AUC (see analysis), because models of
    different size react differently to these out-of-distribution inputs."""
    _, _, H, W = x.shape
    order = torch.as_tensor(order, device=x.device)
    bl = blur(x)
    counts = torch.linspace(0, H * W, steps + 1).long()
    dele, ins = [], []
    for c in counts:
        m = torch.zeros(H * W, device=x.device)
        m[order[:c]] = 1
        m = m.view(1, 1, H, W)
        dele.append(x * (1 - m))
        ins.append(bl * (1 - m) + x * m)
    p = _probs(model, torch.cat(dele + ins), target, amp)
    d, n = p[:steps + 1], p[steps + 1:]
    return dict(deletion_auc=float(_trapz(d, dx=1 / steps)), insertion_auc=float(_trapz(n, dx=1 / steps)))


def saliency_order(sal, seed=0):
    """Most-salient-first pixel order; exact ties (blocky upsampled CAMs) broken at random, reproducibly."""
    rng = np.random.default_rng(seed)
    flat = sal.ravel().astype(np.float64)
    return np.lexsort((rng.random(flat.size), -flat))


# ---------------------------------------------------------------- lesion reliance (model property, no XAI)
def _dilate(mask_t, px):
    return F.max_pool2d(mask_t, 2 * px + 1, stride=1, padding=px) if px > 0 else mask_t


def _control_region(m, shifts):
    """Same-shape region shifted to overlap the lesion as little as possible (torch.roll wrap-around)."""
    best, best_ov = None, 2.0
    tot = float(m.sum()) + 1e-8
    for dy, dx in shifts:
        c = torch.roll(m, shifts=(dy, dx), dims=(-2, -1))
        ov = float((c * m).sum()) / tot
        if ov < best_ov:
            best, best_ov = c, ov
    return best, best_ov


@torch.no_grad()
def lesion_reliance(model, x, target, mask, dilate_px=8, amp=True):
    """How much the model's confidence depends on the lesion region itself (occlusion test, no XAI).
    drop_lesion : p(orig) - p(lesion replaced by its blurred version)
    drop_ctrl   : same for an equally large region elsewhere (controls for the size of the perturbation)
    lri         : drop_lesion - drop_ctrl  (> 0: the model relies on the lesion more than on other tissue)
    p_lesion_only : p when everything outside the (dilated) lesion is blurred."""
    _, _, H, W = x.shape
    m = torch.from_numpy(mask.astype(np.float32)).to(x.device)[None, None]
    m = _dilate(m, dilate_px)
    shifts = [(int(H * a), int(W * b)) for a in (0, .25, .5, .75) for b in (0, .25, .5, .75) if (a, b) != (0, 0)]
    c, ov = _control_region(m, shifts)
    bl = blur(x)
    batch = torch.cat([x, bl * m + x * (1 - m), bl * c + x * (1 - c), x * m + bl * (1 - m)])
    p = _probs(model, batch, target, amp)
    return dict(p_orig=float(p[0]), drop_lesion=float(p[0] - p[1]), drop_ctrl=float(p[0] - p[2]),
                lri=float(p[2] - p[1]), p_lesion_only=float(p[3]), ctrl_overlap=float(ov),
                dilated_area=float(m.mean()))


# ---------------------------------------------------------------- cascading sanity check
def randomization_levels(model):
    """Top-down cascade (Adebayo et al., 2018), identical level names for S, M and L."""
    b = model.blocks
    return [("classifier", [model.classifier]),
            ("head", [model.conv_head, model.bn2]),
            ("stage-1", [b[-1]]),
            ("stage-2", [b[-2]]),
            ("stage-3", [b[-3]]),
            ("all", list(b[:-3]) + [model.conv_stem, model.bn1])]


@torch.no_grad()
def _randomize(mods, g):
    for mod in mods:
        for p in mod.parameters():
            scale = float(p.float().std()) if p.numel() > 1 else 1.0
            p.copy_(torch.randn(p.shape, generator=g).to(p) * (scale + 1e-3))


def cascade_models(model, seed=0):
    """Yields (level_index, level_name, randomized deep copy) with cumulative randomization."""
    m = copy.deepcopy(model).eval()
    g = torch.Generator().manual_seed(seed)
    for i, (name, _) in enumerate(randomization_levels(model)):
        mods = dict(randomization_levels(m))[name]
        _randomize(mods, g)
        yield i, name, m


def map_rank_corr(a, b, size=48):
    """Spearman correlation between the trained-model map and the randomized-model map.
    A constant randomized map means the explanation was destroyed: scored as 0."""
    ta = F.adaptive_avg_pool2d(torch.from_numpy(np.ascontiguousarray(a))[None, None].float(), size).ravel().numpy()
    tb = F.adaptive_avg_pool2d(torch.from_numpy(np.ascontiguousarray(b))[None, None].float(), size).ravel().numpy()
    if np.ptp(ta) == 0:
        return np.nan
    if np.ptp(tb) == 0 or not np.isfinite(tb).all():
        return 0.0
    r = spearmanr(ta, tb).correlation
    return float(r) if np.isfinite(r) else np.nan


# ---------------------------------------------------------------- effective receptive field
def effective_receptive_field(model, loader, device, n_images=32, mass=0.9):
    """Luo et al. (2016): gradient of the centre unit of the last feature map w.r.t. the input.
    Model-level descriptive quantity (one value per trained model), NOT an image-level predictor.
    Returns (erf_map HxW, fraction of image area holding `mass` of the gradient magnitude)."""
    model.eval()
    acc, seen = None, 0
    for x, *_ in loader:
        for i in range(x.shape[0]):
            xi = x[i:i + 1].to(device).requires_grad_(True)
            f = model.forward_features(xi)
            h, w = f.shape[-2:]
            f[0, :, h // 2, w // 2].sum().backward()
            gmap = xi.grad.detach().abs().sum(1)[0].float().cpu().numpy()
            acc = gmap if acc is None else acc + gmap
            seen += 1
            model.zero_grad(set_to_none=True)
            if seen >= n_images:
                break
        if seen >= n_images:
            break
    erf = acc / seen
    v = np.sort(erf.ravel())[::-1]
    cum = np.cumsum(v) / (v.sum() + 1e-12)
    area = float((np.searchsorted(cum, mass) + 1) / v.size)
    return erf.astype(np.float32), area

### Modul `train.py`: satu run training + evaluasi + XAI

In [ ]:
%%writefile train.py
"""One training run: dataset x variant x split x seed x image size -> metrics, XAI scores, ERF, efficiency.

Usage (one process per GPU; the notebook launches these):
    CUDA_VISIBLE_DEVICES=0 python train.py --dataset ham10000 --variant s --run fixed --seed 1 ...

Controls that keep S, M and L comparable (see methodology v2, section 4):
  * weights: timm tf_efficientnetv2_{s,m,l}.in1k, all ported from the same official Google checkpoints
    (same ImageNet-1k recipe) and normalized with each model's own pretrained_cfg mean/std;
  * BatchNorm running statistics frozen (bn_mode=frozen), so the per-step micro-batch (32/16/8) does not
    change the maths: with gradient accumulation to an effective batch of 32, every variant gets the
    same update rule;
  * same optimizer, schedule, augmentation, loss, early stopping and learning-rate grid.

Stability (v3, after the pilot where L diverged at lr 3e-4: NaN loss on HAM10000, one-class collapse on BUSI):
  * classifier initialised to zero, so the first steps train the head before large gradients reach the backbone;
  * warm-up of max(1 epoch, --warmup_steps) optimizer steps (capped at 1/4 of training) instead of 1 epoch;
  * non-finite losses are skipped; a run with too many is stopped as "diverged@<epoch>";
  * a run whose best checkpoint predicts a single class on validation is marked "collapsed";
    diverged/collapsed runs skip XAI and are excluded by analysis.py (status filter).
  The same rules apply to every variant.

v3.1 (review of v3):
  * --max_bad_loss lowered from 0.2 to 0.02: skipping many non-finite batches would silently train one variant
    on less data than the others; the run-level totals (bad_loss_frac_total / _max_epoch) go to summary.json and
    are reported per variant by analysis.py;
  * --attempt / amp recorded, so runs re-done under the pre-specified rescue protocol (notebook section 5) can be
    identified and reported (and excluded in a sensitivity analysis).
"""
import argparse
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T

import xai_utils as X

WEIGHTS = {"s": "tf_efficientnetv2_s.in1k", "m": "tf_efficientnetv2_m.in1k", "l": "tf_efficientnetv2_l.in1k"}
# (batch per step, accumulation steps) -> effective batch 32 on a 16 GB T4 (FP16)
BATCH = {384: {"s": (32, 1), "m": (16, 2), "l": (8, 4)},
         480: {"s": (16, 2), "m": (8, 4), "l": (4, 8)}}
NO_FLIP = {"siim"}  # chest X-ray: horizontal flip moves the heart to the wrong side


def parse():
    a = argparse.ArgumentParser()
    a.add_argument("--dataset", required=True)
    a.add_argument("--variant", required=True, choices=["s", "m", "l"])
    a.add_argument("--run", required=True, help="split id: fold number (busi) or 'fixed'")
    a.add_argument("--seed", type=int, default=0)
    a.add_argument("--cache", required=True)
    a.add_argument("--out", required=True)
    a.add_argument("--img_size", type=int, default=384)
    a.add_argument("--epochs", type=int, default=30)
    a.add_argument("--patience", type=int, default=7)
    a.add_argument("--lr", type=float, default=3e-4)
    a.add_argument("--weight_decay", type=float, default=0.05)
    a.add_argument("--drop_rate", type=float, default=0.2)
    a.add_argument("--warmup_steps", type=int, default=100, help="minimum warm-up optimizer steps")
    a.add_argument("--max_bad_loss", type=float, default=0.02,
                   help="stop as diverged if this fraction of an epoch's batches has a non-finite loss")
    a.add_argument("--no_amp", action="store_true", help="train in FP32 (slower; fallback if FP16 overflows)")
    a.add_argument("--attempt", type=int, default=0, help="0 = planned run; 1, 2 = rescue attempts (see notebook)")
    a.add_argument("--bn_mode", default="frozen", choices=["frozen", "train"])
    a.add_argument("--grad_ckpt", action="store_true", help="gradient checkpointing (if L runs out of memory)")
    a.add_argument("--train_frac", type=float, default=1.0)
    a.add_argument("--tag", default="main", help="analysis group: main | native | datafrac")
    a.add_argument("--shard", default="", help="which collaborator account ran this job (traceability only)")
    a.add_argument("--xai_n", type=int, default=300, help="lesion images scored with CAM/IG")
    a.add_argument("--scorecam_n", type=int, default=0)
    a.add_argument("--insdel_n", type=int, default=100)
    a.add_argument("--reliance_n", type=int, default=300)
    a.add_argument("--sanity_n", type=int, default=30)
    a.add_argument("--sanity_methods", default="gradcam,ig")
    a.add_argument("--erf_n", type=int, default=32)
    a.add_argument("--methods", default="gradcam,gradcampp,ig")
    a.add_argument("--no_xai", action="store_true", help="used by the learning-rate search")
    a.add_argument("--deadline", type=float, default=0, help="unix time; stop training if exceeded")
    a.add_argument("--keep_ckpt", action="store_true")
    a.add_argument("--no_pretrained", action="store_true", help="offline smoke test only")
    a.add_argument("--workers", type=int, default=2)
    a.add_argument("--max_steps", type=int, default=0, help="smoke test: cap optimizer steps per epoch")
    a.add_argument("--ig_steps", type=int, default=32)
    a.add_argument("--insdel_steps", type=int, default=25)
    return a.parse_args()


class ImgDS(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df.reset_index(drop=True), tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tf(T.functional.to_image(Image.open(r.image).convert("RGB")))
        return x, int(r.label), i


def transforms(train, size, mean, std, flip=True):
    base = [T.Resize((size, size), antialias=True)]
    aug = ([T.RandomHorizontalFlip()] if flip else []) + \
          [T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)), T.ColorJitter(0.1, 0.1)]
    return T.Compose(base + (aug if train else []) + [T.ToDtype(torch.float32, scale=True), T.Normalize(mean, std)])


def build_model(variant, n_classes, pretrained, drop_rate=0.2):
    m = timm.create_model(WEIGHTS[variant], pretrained=pretrained, num_classes=n_classes, drop_rate=drop_rate)
    head = m.get_classifier()
    nn.init.zeros_(head.weight)
    nn.init.zeros_(head.bias)
    cfg = m.pretrained_cfg
    return m, list(cfg["mean"]), list(cfg["std"])


def freeze_bn_stats(model):
    """Keep ImageNet running mean/var (affine gamma/beta stay trainable)."""
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.eval()


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, loader, device, amp):
    model.eval()
    P, Y = [], []
    for x, y, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            P.append(model(x.to(device, non_blocking=True).to(memory_format=torch.channels_last))
                     .float().softmax(1).cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def cls_metrics(p, y, n_classes):
    pred = p.argmax(1)
    out = dict(acc=accuracy_score(y, pred), bal_acc=balanced_accuracy_score(y, pred),
               macro_f1=f1_score(y, pred, average="macro", labels=list(range(n_classes)), zero_division=0))
    try:
        out["auroc"] = roc_auc_score(y, p[:, 1]) if n_classes == 2 else \
            roc_auc_score(y, p, multi_class="ovr", average="macro", labels=list(range(n_classes)))
    except ValueError:
        out["auroc"] = float("nan")
    return {k: float(v) for k, v in out.items()}


def main():
    a = parse()
    t_start = time.time()
    seed_all(a.seed)
    os.makedirs(a.out, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = device == "cuda" and not a.no_amp
    torch.backends.cudnn.benchmark = True

    ds_dir = os.path.join(a.cache, a.dataset)
    size_dir = os.path.join(ds_dir, str(a.img_size))
    df = pd.read_csv(os.path.join(ds_dir, "index.csv"), keep_default_na=False)
    ms = pd.read_csv(os.path.join(size_dir, "masks.csv"))
    df = df.merge(ms, on="uid", how="left")
    df["image"] = [os.path.join(size_dir, "img", f"{u}.jpg") for u in df.uid]
    df["mask"] = [os.path.join(size_dir, "mask", f"{u}.png") if (ar == ar and ar > 0) else ""
                  for u, ar in zip(df.uid, df.mask_area)]
    split = json.load(open(os.path.join(ds_dir, "splits.json")))[a.run]
    n_classes = int(df.label.max()) + 1
    by = df.set_index("uid")
    tr, va, te = (by.loc[split[k]].reset_index() for k in ("train", "val", "test"))
    if a.train_frac < 1.0:
        tr = tr.groupby("label", group_keys=False).apply(
            lambda g: g.sample(frac=a.train_frac, random_state=a.seed))

    model, mean, std = build_model(a.variant, n_classes, not a.no_pretrained, a.drop_rate)
    if a.grad_ckpt:
        model.set_grad_checkpointing(True)
    model = model.to(device).to(memory_format=torch.channels_last)
    flip = a.dataset not in NO_FLIP

    bs, accum = BATCH.get(a.img_size, BATCH[384])[a.variant]
    gen = torch.Generator().manual_seed(a.seed)
    dl_tr = DataLoader(ImgDS(tr, transforms(True, a.img_size, mean, std, flip)), bs, shuffle=True,
                       drop_last=len(tr) > bs * accum, num_workers=a.workers, pin_memory=True, generator=gen,
                       persistent_workers=a.workers > 0)
    mk = lambda d, b=32: DataLoader(ImgDS(d, transforms(False, a.img_size, mean, std)), b,
                                    num_workers=a.workers, pin_memory=True)
    dl_va, dl_te = mk(va), mk(te)

    counts = np.bincount(tr.label, minlength=n_classes).astype(float)
    w = torch.tensor(counts.sum() / (n_classes * np.maximum(counts, 1)), dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=a.lr, weight_decay=a.weight_decay)
    steps_per_epoch = math.ceil(len(dl_tr) / accum) if not a.max_steps else a.max_steps
    total = a.epochs * steps_per_epoch
    warm = max(1, min(max(steps_per_epoch, a.warmup_steps), total // 4))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    tracker = None
    if device == "cuda":
        try:
            from codecarbon import EmissionsTracker
            tracker = EmissionsTracker(output_dir=a.out, log_level="error", save_to_file=True,
                                       tracking_mode="process",
                                       gpu_ids=[int(os.environ.get("PHYSICAL_GPU", 0))])
            tracker.start()
        except Exception as e:  # energy logging is optional
            print("codecarbon disabled:", e)

    ckpt = os.path.join(a.out, "best.pt")
    best, bad, hist, n_seen, t_train = -1.0, 0, [], 0, 0.0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    stopped, best_n_pred = "completed", 0
    tot_bad, tot_batches, max_bad_ep = 0, 0, 0.0
    for ep in range(a.epochs):
        model.train()
        if a.bn_mode == "frozen":
            freeze_bn_stats(model)
        t0, loss_sum, nb, n_bad, has_grad = time.time(), 0.0, 0, 0, False
        opt.zero_grad(set_to_none=True)
        for i, (x, y, _) in enumerate(dl_tr):
            if a.max_steps and i >= a.max_steps * accum:
                break
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = crit(model(x).float(), y) / accum
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                has_grad = True
                loss_sum += loss.item() * accum
                nb += 1
            else:
                n_bad += 1
            if (i + 1) % accum == 0 or i + 1 == len(dl_tr):
                if has_grad:
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    scaler.step(opt)
                    scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                has_grad = False
            n_seen += x.shape[0]
        if device == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - t0
        t_train += dt
        p, y = predict(model, dl_va, device, amp)
        m = cls_metrics(p, y, n_classes)
        n_pred = len(np.unique(p.argmax(1))) if np.isfinite(p).all() else 0
        bad_frac = n_bad / max(n_bad + nb, 1)
        tot_bad, tot_batches, max_bad_ep = tot_bad + n_bad, tot_batches + n_bad + nb, max(max_bad_ep, bad_frac)
        hist.append(dict(epoch=ep, loss=loss_sum / max(nb, 1), epoch_sec=dt, lr=opt.param_groups[0]["lr"],
                         bad_loss_frac=bad_frac, val_n_pred_classes=n_pred,
                         **{f"val_{k}": v for k, v in m.items()}))
        pd.DataFrame(hist).to_csv(os.path.join(a.out, "history.csv"), index=False)
        print(f"[{a.dataset}/{a.variant}/{a.run}/s{a.seed}/{a.img_size}] ep {ep} loss {hist[-1]['loss']:.4f} "
              f"val_f1 {m['macro_f1']:.4f} pred_classes {n_pred}/{n_classes} bad_loss {bad_frac:.2f} ({dt:.0f}s)",
              flush=True)
        weights_ok = all(torch.isfinite(q).all() for q in model.parameters())
        if bad_frac > a.max_bad_loss or n_pred == 0 or not weights_ok:
            stopped = f"diverged@{ep}"
            print(f"DIVERGED at epoch {ep} (non-finite loss fraction {bad_frac:.2f}, finite weights {weights_ok}); "
                  "try a lower --lr or --no_amp", flush=True)
            if best < 0:   # no usable checkpoint yet: keep the current weights so the test/XAI code still runs
                torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, ckpt)
            break
        if m["macro_f1"] > best:
            best, bad, best_n_pred = m["macro_f1"], 0, n_pred
            torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, ckpt)
        else:
            bad += 1
            if bad >= a.patience:
                stopped = f"early_stop@{ep}"
                break
        if a.deadline and time.time() > a.deadline:
            stopped = f"deadline@{ep}"
            break

    energy = None
    if tracker is not None:
        try:
            tracker.stop()
            energy = float(tracker.final_emissions_data.energy_consumed)  # kWh
        except Exception:
            pass
    peak_mem = torch.cuda.max_memory_allocated() / 2 ** 30 if device == "cuda" else float("nan")

    sd = torch.load(ckpt, map_location=device)
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    model.eval()
    p, y = predict(model, dl_te, device, amp)
    test = cls_metrics(p, y, n_classes)
    pd.DataFrame({"uid": te.uid, "label": y, "pred": p.argmax(1), **{f"p{c}": p[:, c] for c in range(n_classes)}}) \
        .to_csv(os.path.join(a.out, "test_predictions.csv"), index=False)

    summary = dict(tag=a.tag, shard=a.shard, dataset=a.dataset, variant=a.variant, weights=WEIGHTS[a.variant], run=a.run, seed=a.seed,
                   lr=a.lr, img_size=a.img_size, bn_mode=a.bn_mode, train_frac=a.train_frac, n_train=len(tr),
                   n_val=len(va), n_test=len(te), status=stopped, epochs_run=len(hist), best_val_macro_f1=best,
                   train_sec=t_train, train_img_per_sec=n_seen / max(t_train, 1e-6), peak_train_mem_gb=peak_mem,
                   energy_kwh=energy, params_m=sum(q.numel() for q in model.parameters()) / 1e6,
                   batch=bs, accum=accum, attempt=a.attempt, amp=bool(amp),
                   bad_loss_frac_total=tot_bad / max(tot_batches, 1), bad_loss_frac_max_epoch=max_bad_ep,
                   **{f"test_{k}": v for k, v in test.items()})

    if not stopped.startswith(("diverged", "deadline")) and best_n_pred <= 1:
        stopped = f"collapsed({stopped})"
        print("COLLAPSED: best checkpoint predicts a single class on validation; excluded from analysis", flush=True)
    summary["status"] = stopped
    failed = stopped.startswith(("diverged", "collapsed"))
    if not a.no_xai and not failed:
        summary.update(run_xai(a, model, te, device, amp, mean, std))
    summary["total_sec"] = time.time() - t_start
    json.dump(summary, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    if not a.keep_ckpt:
        os.remove(ckpt)
    if stopped.startswith("deadline"):
        print("Stopped by deadline; run will be repeated next session.")
    else:
        open(os.path.join(a.out, "DONE"), "w").write(stopped)
    print(json.dumps({k: summary[k] for k in ("test_macro_f1", "test_auroc", "train_img_per_sec")}), flush=True)


def run_xai(a, model, te, device, amp, mean, std):
    """Explanation quality on test images that contain a lesion, target = true class.
    The same images (fixed shuffle) are used for S, M and L, so every score is paired by image.
    Correctness is stored so the analysis can restrict to images all variants classify correctly."""
    methods = [m for m in a.methods.split(",") if m]
    if a.scorecam_n > 0 and "scorecam" not in methods:
        methods.append("scorecam")
    les = te[te["mask"] != ""].copy()
    les = les.sample(frac=1.0, random_state=0).head(a.xai_n).reset_index(drop=True)
    tf = transforms(False, a.img_size, mean, std)
    for q in model.parameters():
        q.requires_grad_(True)
    ex = X.make_explainers(model, methods)
    grid = a.img_size // 32
    rows, rel_rows, san_rows = [], [], []
    t0 = time.time()
    xs, masks = [], []
    for i, r in les.iterrows():
        x = tf(T.functional.to_image(Image.open(r.image).convert("RGB")))[None].to(device)
        mask = np.asarray(Image.open(r["mask"]).convert("L").resize((a.img_size, a.img_size), Image.NEAREST)) > 127
        with torch.no_grad():
            prob = model(x).float().softmax(1)[0]
        base = dict(uid=r.uid, label=int(r.label), pred=int(prob.argmax()), p_true=float(prob[int(r.label)]))
        base["correct"] = int(base["pred"] == base["label"])
        maps = {}
        for name, e in ex.items():
            if name == "scorecam" and i >= a.scorecam_n:
                continue
            maps.update(X.saliency(name, e, x, r.label, a.ig_steps))
        refs = X.reference_maps(mask, grid, seed=i)
        rnd_order = None
        for name, s in list(maps.items()) + list(refs.items()):
            row = dict(base, method=name, **X.localization_metrics(s, mask))
            if i < a.insdel_n and name in maps:
                row.update(X.insertion_deletion(model, x, r.label, X.saliency_order(s, seed=i), a.insdel_steps, amp))
                if rnd_order is None:  # random pixel order: per-image reference for faithfulness
                    rnd_order = np.random.default_rng(10_000 + i).permutation(s.size)
                    rnd = X.insertion_deletion(model, x, r.label, rnd_order, a.insdel_steps, amp)
                row.update(deletion_auc_random=rnd["deletion_auc"], insertion_auc_random=rnd["insertion_auc"])
            rows.append(row)
        if i < a.reliance_n:
            rel_rows.append(dict(base, mask_area=float(mask.mean()),
                                 **X.lesion_reliance(model, x, r.label, mask, amp=amp)))
        if i < a.sanity_n:
            xs.append((x, r, {k: maps[k] for k in maps}))
    X.release(ex)
    pd.DataFrame(rows).to_csv(os.path.join(a.out, "xai_per_image.csv"), index=False)
    pd.DataFrame(rel_rows).to_csv(os.path.join(a.out, "reliance_per_image.csv"), index=False)
    t_xai = time.time() - t0

    # cascading model-parameter randomization (Adebayo et al., 2018)
    san_methods = [m for m in a.sanity_methods.split(",") if m in methods]
    if xs and san_methods:
        for lvl, lname, rm in X.cascade_models(model, seed=a.seed):
            exr = X.make_explainers(rm, san_methods)
            for x, r, maps in xs:
                for name, e in exr.items():
                    for k, s_r in X.saliency(name, e, x, r.label, a.ig_steps).items():
                        san_rows.append(dict(uid=r.uid, method=k, level=lvl, level_name=lname,
                                             rank_corr=X.map_rank_corr(maps[k], s_r)))
            X.release(exr)
            del exr
    pd.DataFrame(san_rows).to_csv(os.path.join(a.out, "sanity_per_image.csv"), index=False)

    erf_loader = DataLoader(ImgDS(les.head(a.erf_n) if len(les) else te.head(a.erf_n), tf), 8)
    erf, erf_area = X.effective_receptive_field(model, erf_loader, device, a.erf_n)
    np.save(os.path.join(a.out, "erf.npy"), erf)

    d = pd.DataFrame(rows)
    out = dict(xai_n_images=int(les.shape[0]), xai_sec=t_xai, sanity_sec=time.time() - t0 - t_xai,
               erf_area90=erf_area)
    if len(d):
        for mth, g in d.groupby("method"):
            for c in ("hit", "energy_norm", "iou_topk", "rra", "insertion_auc", "deletion_auc"):
                if c in g:
                    out[f"xai_{mth}_{c}"] = float(g[c].mean())
    if rel_rows:
        out["reliance_lri"] = float(pd.DataFrame(rel_rows).lri.mean())
    return out


if __name__ == "__main__":
    main()

### Modul `bench.py`: benchmark efisiensi terkontrol

In [ ]:
%%writefile bench.py
"""Efficiency benchmark per variant and image size, run alone on one GPU (no other job running).

Reported per variant:
  params_m, gmacs (multiply-accumulates; FlopCounterMode counts 1 MAC = 2 FLOPs, so gmacs = flops / 2,
  the unit used in the EfficientNetV2 paper), GPU latency (batch 1, FP16), GPU throughput (batch 32, FP16),
  CPU latency (batch 1, FP32), and a *controlled* training step: fixed batch 8, FP16, frozen BN, so peak
  training memory and training throughput are comparable across variants (the in-run numbers are not:
  they use different micro-batches and share 4 vCPUs with a parallel run)."""
import argparse
import json
import time

import timm
import torch
import torch.nn as nn
from torch.utils.flop_counter import FlopCounterMode

WEIGHTS = {"s": "tf_efficientnetv2_s.in1k", "m": "tf_efficientnetv2_m.in1k", "l": "tf_efficientnetv2_l.in1k"}


def timed(fn, warmup, iters, cuda):
    for _ in range(warmup):
        fn()
    if cuda:
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters):
        fn()
    if cuda:
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters


def main():
    a = argparse.ArgumentParser()
    a.add_argument("--variants", default="s,m,l")
    a.add_argument("--img_sizes", default="384")
    a.add_argument("--n_classes", type=int, default=2)
    a.add_argument("--out", required=True)
    a.add_argument("--cpu_iters", type=int, default=20)
    a.add_argument("--gpu_iters", type=int, default=200)
    a.add_argument("--train_batch", type=int, default=8)
    a = a.parse_args()
    cuda = torch.cuda.is_available()
    torch.set_num_threads(4)
    res = []
    for size in [int(s) for s in a.img_sizes.split(",")]:
        for v in a.variants.split(","):
            m = timm.create_model(WEIGHTS[v], pretrained=False, num_classes=a.n_classes).eval()
            x1 = torch.randn(1, 3, size, size)
            with FlopCounterMode(display=False) as fc, torch.no_grad():
                m(x1)
            r = dict(variant=v, img_size=size, params_m=sum(p.numel() for p in m.parameters()) / 1e6,
                     gmacs=fc.get_total_flops() / 2e9)
            with torch.no_grad():
                r["cpu_latency_ms"] = 1000 * timed(lambda: m(x1), 3, a.cpu_iters, False)
            if cuda:
                g = m.cuda().to(memory_format=torch.channels_last)
                xg1 = x1.cuda().to(memory_format=torch.channels_last)
                xg32 = torch.randn(32, 3, size, size, device="cuda").to(memory_format=torch.channels_last)
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                    r["gpu_latency_ms"] = 1000 * timed(lambda: g(xg1), 20, a.gpu_iters, True)
                    torch.cuda.reset_peak_memory_stats()
                    r["gpu_throughput_img_s"] = 32 / timed(lambda: g(xg32), 5, 30, True)
                    r["gpu_infer_mem_gb_b32"] = torch.cuda.max_memory_allocated() / 2 ** 30
                # controlled training step
                g.train()
                for mod in g.modules():
                    if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                        mod.eval()
                opt = torch.optim.AdamW(g.parameters(), lr=1e-4)
                scaler = torch.amp.GradScaler("cuda")
                xb = torch.randn(a.train_batch, 3, size, size, device="cuda").to(memory_format=torch.channels_last)
                yb = torch.randint(0, a.n_classes, (a.train_batch,), device="cuda")

                def step():
                    with torch.autocast("cuda", dtype=torch.float16):
                        loss = nn.functional.cross_entropy(g(xb), yb)
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)

                try:
                    torch.cuda.reset_peak_memory_stats()
                    dt = timed(step, 3, 20, True)
                    r[f"train_img_s_b{a.train_batch}"] = a.train_batch / dt
                    r[f"train_mem_gb_b{a.train_batch}"] = torch.cuda.max_memory_allocated() / 2 ** 30
                except torch.cuda.OutOfMemoryError:
                    r[f"train_img_s_b{a.train_batch}"] = float("nan")
                    r[f"train_mem_gb_b{a.train_batch}"] = float("nan")
                r["gpu_name"] = torch.cuda.get_device_name(0)
                del g, opt
                torch.cuda.empty_cache()
            print(r, flush=True)
            res.append(r)
    json.dump(res, open(a.out, "w"), indent=2)


if __name__ == "__main__":
    main()

### Modul `analysis.py`: analisis statistik yang sudah ditetapkan sebelumnya

In [ ]:
%%writefile analysis.py
"""Pre-specified statistical analysis (methodology v2, section 6).

Reads every finished run under --results and writes CSV tables + analysis_report.txt to --out.

Estimation units
  * performance: test images (paired across variants), metric averaged over the repetitions of a variant
    (5 seeds on the fixed split for HAM10000/SIIM; pooled out-of-fold predictions for BUSI).
  * explanations: one score per image and variant (mean over repetitions), paired across variants.
Inference
  * paired bootstrap over images (B resamples) -> 95% CI, two-sided bootstrap p; 90% CI for TOST
    equivalence against the pre-registered margins; Wilcoxon signed-rank on per-image differences;
    paired dominance d = P(A > B) - P(A < B) as effect size; Holm correction within each family.
  * H4 (lesion size): linear mixed model  score ~ variant * size_tertile + (1 | image),
    likelihood-ratio test of the interaction (ML fit).
"""
import argparse
import glob
import json
import os
import warnings

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

PAIRS = [("m", "s"), ("l", "s"), ("l", "m")]
# pre-registered equivalence margins (change ONLY before the full experiment, and report them)
MARGINS = {"macro_f1": 0.03, "bal_acc": 0.03, "auroc": 0.02,
           "energy_norm": 0.05, "hit": 0.05, "iou_topk": 0.05, "del_gain": 0.05, "ins_gain": 0.05,
           "lri": 0.05, "drop_lesion": 0.05, "rra": 0.05, "dod": 0.05}
PRIMARY_XAI = ("gradcam", "energy_norm")
# RQ6 (method dependence): pre-specified contrast = Grad-CAM vs IG on energy_norm; the other pairs are sensitivity
RQ6_PRIMARY = (("gradcam", "ig"), "energy_norm")
RQ6_PAIRS = [("gradcam", "ig"), ("gradcam", "ig_raw"), ("gradcampp", "ig")]


# ---------------------------------------------------------------- helpers
def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m, run = len(idx), 0.0
    for r, i in enumerate(idx):
        run = max(run, min(1.0, (m - r) * p[i]))
        out[i] = run
    return out


def boot_p(bs):
    bs = bs[np.isfinite(bs)]
    if not len(bs):
        return np.nan
    return float(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())))


def summarize_boot(delta, bs, margin):
    lo95, hi95 = np.nanpercentile(bs, [2.5, 97.5])
    lo90, hi90 = np.nanpercentile(bs, [5, 95])
    return dict(delta=delta, ci95_lo=lo95, ci95_hi=hi95, ci90_lo=lo90, ci90_hi=hi90, p_boot=boot_p(bs),
                margin=margin, equivalent=bool(margin is not None and lo90 > -margin and hi90 < margin))


def macro_f1(y, pred, C):
    cm = np.bincount(y * C + pred, minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1)
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return float(f1.mean())


def bal_acc(y, pred, C):
    cm = np.bincount(y * C + pred, minlength=C * C).reshape(C, C)
    rec = np.diag(cm) / np.maximum(cm.sum(1), 1)
    return float(rec[cm.sum(1) > 0].mean())


def auroc(y, P, C):
    try:
        return float(roc_auc_score(y, P[:, 1]) if C == 2 else
                     roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(C))))
    except ValueError:
        return np.nan


METRIC_FN = {"macro_f1": lambda y, P, C: macro_f1(y, P.argmax(1), C),
             "bal_acc": lambda y, P, C: bal_acc(y, P.argmax(1), C),
             "auroc": auroc}


# ---------------------------------------------------------------- loading
def load_runs(results):
    rows = []
    for p in glob.glob(f"{results}/**/summary.json", recursive=True):
        d = json.load(open(p))
        d["dir"] = os.path.dirname(p)
        d["archived"] = "__failed-attempt" in d["dir"]   # earlier failed attempt kept by the rescue protocol
        d.setdefault("tag", "main")
        d.setdefault("attempt", 0)
        rows.append(d)
    S = pd.DataFrame(rows)
    if len(S):
        S["rep"] = np.where(S.dataset == "busi", "fold" + S.run.astype(str), "seed" + S.seed.astype(str))
    return S


def load_per_image(S, fname):
    parts = []
    for r in S.itertuples():
        f = os.path.join(r.dir, fname)
        if os.path.exists(f) and os.path.getsize(f) > 1:
            try:
                d = pd.read_csv(f)
            except pd.errors.EmptyDataError:
                continue
            for k in ("dataset", "variant", "rep", "tag"):
                d[k] = getattr(r, k)
            parts.append(d)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# ---------------------------------------------------------------- performance
def perf_tables(S, n_boot, rng):
    desc = S.groupby(["tag", "dataset", "variant"])[
        [c for c in ["test_macro_f1", "test_bal_acc", "test_auroc", "test_acc", "epochs_run"] if c in S]] \
        .agg(["mean", "std", "count"]).round(4)
    comp = []
    for (tag, ds), g in S.groupby(["tag", "dataset"]):
        preds = {}
        for v, gv in g.groupby("variant"):
            reps = []
            for r in gv.itertuples():
                f = os.path.join(r.dir, "test_predictions.csv")
                if os.path.exists(f):
                    reps.append(pd.read_csv(f))
            if not reps:
                continue
            if ds == "busi":   # folds have different test images: pool out-of-fold predictions
                reps = [pd.concat(reps, ignore_index=True)]
            preds[v] = [r.sort_values("uid").reset_index(drop=True) for r in reps]
        for A, B in PAIRS:
            if A not in preds or B not in preds:
                continue
            uids = sorted(set.intersection(*[set(r.uid) for r in preds[A] + preds[B]]))
            if not uids:
                continue
            pa = [r.set_index("uid").loc[uids] for r in preds[A]]
            pb = [r.set_index("uid").loc[uids] for r in preds[B]]
            pcols = [c for c in pa[0].columns if c.startswith("p") and c[1:].isdigit()]
            C = len(pcols)
            y = pa[0].label.values.astype(int)
            PA = [r[pcols].values for r in pa]
            PB = [r[pcols].values for r in pb]
            for met, fn in METRIC_FN.items():
                val = lambda Ps, idx: np.nanmean([fn(y[idx], P[idx], C) for P in Ps])
                full = np.arange(len(y))
                delta = val(PA, full) - val(PB, full)
                bs = np.empty(n_boot)
                for b in range(n_boot):
                    idx = rng.integers(0, len(y), len(y))
                    bs[b] = val(PA, idx) - val(PB, idx)
                comp.append(dict(tag=tag, dataset=ds, metric=met, A=A, B=B, n_images=len(y),
                                 reps_A=len(PA), reps_B=len(PB), **summarize_boot(delta, bs, MARGINS.get(met))))
    comp = pd.DataFrame(comp)
    if len(comp):
        comp["family"] = np.where(comp.metric == "macro_f1", "primary_perf", "secondary_perf")
        comp["p_holm"] = np.nan
        for (t, fam), idx in comp.groupby(["tag", "family"]).groups.items():
            comp.loc[idx, "p_holm"] = holm(comp.loc[idx, "p_boot"])
    return desc, comp


# ---------------------------------------------------------------- explanations
def add_derived(X):
    X = X.copy()
    if "deletion_auc" in X:
        X["del_gain"] = X["deletion_auc_random"] - X["deletion_auc"]
        X["ins_gain"] = X["insertion_auc"] - X["insertion_auc_random"]
    # RRA for runs made before it was computed directly (pilot v3.1): with |top-k| = |mask| = k,
    # IoU = I / (2k - I)  =>  RRA = I / k = 2 IoU / (1 + IoU)   (exact unless ties sit at the threshold)
    if "iou_topk" in X:
        derived = 2 * X["iou_topk"] / (1 + X["iou_topk"])
        if "rra" not in X:
            X["rra"] = np.nan
        X["rra_derived"] = X["rra"].isna() & derived.notna()
        X["rra"] = X["rra"].fillna(derived)
    return X


def common_correct(X):
    """Keep (dataset, rep, uid) only if every variant present in that repetition classified it correctly."""
    c = X.drop_duplicates(["tag", "dataset", "rep", "uid", "variant"])[["tag", "dataset", "rep", "uid", "variant", "correct"]]
    nvar = c.groupby(["tag", "dataset"]).variant.nunique().rename("nvar")
    agg = c.groupby(["tag", "dataset", "rep", "uid"]).agg(ok=("correct", "min"), n=("variant", "nunique")).reset_index()
    agg = agg.merge(nvar, on=["tag", "dataset"])
    keep = agg[(agg.ok == 1) & (agg.n == agg.nvar)][["tag", "dataset", "rep", "uid"]]
    return X.merge(keep, on=["tag", "dataset", "rep", "uid"])


def paired_tests(X, metrics, methods, n_boot, rng, subset):
    out = []
    for (tag, ds, mth), g in X[X.method.isin(methods)].groupby(["tag", "dataset", "method"]):
        per = g.groupby(["uid", "variant"])[metrics].mean()
        for met in metrics:
            if met not in per or per[met].isna().all():
                continue
            w = per[met].unstack("variant")
            for A, B in PAIRS:
                if A not in w or B not in w:
                    continue
                d = (w[A] - w[B]).dropna().values
                if len(d) < 5:
                    continue
                bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
                try:
                    pw = float(wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
                except ValueError:
                    pw = np.nan
                out.append(dict(tag=tag, dataset=ds, method=mth, metric=met, subset=subset, A=A, B=B, n_images=len(d),
                                mean_A=float(w[A].mean()), mean_B=float(w[B].mean()),
                                dominance=float(np.mean(np.sign(d))), p_wilcoxon=pw,
                                **summarize_boot(float(d.mean()), bs, MARGINS.get(met))))
    return pd.DataFrame(out)


def _tertile(s):
    if s.nunique() < 3:
        return pd.Series(["medium"] * len(s), index=s.index)
    return pd.qcut(s.rank(method="first"), 3, labels=["small", "medium", "large"]).astype(str)


def add_size_tertiles(X):
    """Lesion-size tertiles per dataset, computed on ALL lesion images (thresholds do not depend on
    which images a variant classifies correctly)."""
    u = X.drop_duplicates(["tag", "dataset", "uid"])[["tag", "dataset", "uid", "mask_area"]].copy()
    u["size_t"] = u.groupby(["tag", "dataset"]).mask_area.transform(_tertile)
    return X.merge(u[["tag", "dataset", "uid", "size_t"]], on=["tag", "dataset", "uid"])


def lesion_size_lmm(X, method="gradcam", metric="energy_norm", min_images=15):
    import statsmodels.formula.api as smf
    from scipy.stats import chi2
    rows = []
    d0 = X[X.method == method]
    for (tag, ds), g in d0.groupby(["tag", "dataset"]):
        g = g.dropna(subset=[metric])
        if g.variant.nunique() < 2 or g.uid.nunique() < min_images:
            continue
        g = g.assign(size_t=g.size_t.astype(str))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                full = smf.mixedlm(f"{metric} ~ C(variant, Treatment('s')) * C(size_t, Treatment('small'))",
                                   g, groups=g["uid"]).fit(reml=False)
                red = smf.mixedlm(f"{metric} ~ C(variant, Treatment('s')) + C(size_t, Treatment('small'))",
                                  g, groups=g["uid"]).fit(reml=False)
            lr = 2 * (full.llf - red.llf)
            dfd = len(full.fe_params) - len(red.fe_params)
            coefs = pd.DataFrame({"coef": full.fe_params, "se": full.bse_fe,
                                  "p": full.pvalues[full.fe_params.index]})
            rows.append(dict(tag=tag, dataset=ds, method=method, metric=metric, n_obs=len(g),
                             n_images=g.uid.nunique(), converged=bool(full.converged and red.converged),
                             lrt_interaction=lr, df=dfd, p_interaction=float(chi2.sf(lr, dfd)),
                             coefs=coefs.round(4).to_dict("index")))
        except Exception as e:  # singular fits on tiny pilots
            rows.append(dict(tag=tag, dataset=ds, method=method, metric=metric, error=str(e)[:200]))
    return pd.DataFrame(rows)


def baseline_table(X):
    cols = [c for c in ["energy_norm", "hit", "iou_topk", "rra", "del_gain", "ins_gain"] if c in X]
    t = X.groupby(["tag", "dataset", "variant", "method"])[cols].mean().round(4)
    # share of images where the method beats the centre-bias baseline (energy_norm)
    piv = X.pivot_table(index=["tag", "dataset", "variant", "rep", "uid"], columns="method", values="energy_norm")
    if "base_center" in piv:
        beat = {m: (piv[m] > piv["base_center"]).groupby(level=[0, 1, 2]).mean()
                for m in piv.columns if not m.startswith(("base_", "oracle"))}
        beat = pd.DataFrame(beat).stack().rename("frac_beats_center").reset_index() \
            .rename(columns={"level_3": "method"})
        t = t.reset_index().merge(beat, on=["tag", "dataset", "variant", "method"], how="left") \
            .set_index(["tag", "dataset", "variant", "method"])
    return t


# ---------------------------------------------------------------- main
def method_dependence(X, n_boot, rng, subset, metrics=("energy_norm", "rra")):
    """RQ6: does the size effect depend on the explainer?  Per image:
    dod = (A - B) under method m1  -  (A - B) under method m2,  e.g. (L - S)_Grad-CAM - (L - S)_IG.
    dod < 0: the larger model looks worse with m1 than with m2 (explainer-specific penalty)."""
    out = []
    for (tag, ds), g in X.groupby(["tag", "dataset"]):
        for met in metrics:
            if met not in g:
                continue
            w = g.groupby(["uid", "method", "variant"])[met].mean().unstack(["method", "variant"])
            for m1, m2 in RQ6_PAIRS:
                for A, B in PAIRS:
                    cols = [(m1, A), (m1, B), (m2, A), (m2, B)]
                    if not all(c in w for c in cols):
                        continue
                    d1 = w[(m1, A)] - w[(m1, B)]
                    d2 = w[(m2, A)] - w[(m2, B)]
                    d = (d1 - d2).dropna().values
                    if len(d) < 5:
                        continue
                    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
                    try:
                        pw = float(wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
                    except ValueError:
                        pw = np.nan
                    prim = (m1, m2) == RQ6_PRIMARY[0] and met == RQ6_PRIMARY[1] and subset == "common_correct"
                    out.append(dict(tag=tag, dataset=ds, metric=met, m1=m1, m2=m2, A=A, B=B, subset=subset,
                                    n_images=len(d), diff_m1=float(np.nanmean(d1)), diff_m2=float(np.nanmean(d2)),
                                    dominance=float(np.mean(np.sign(d))), p_wilcoxon=pw,
                                    family="rq6" if prim else "rq6_sensitivity",
                                    **summarize_boot(float(d.mean()), bs, MARGINS["dod"])))
    return pd.DataFrame(out)


def _direction(row):
    """'higher' / 'lower' (95% CI excludes 0 and Holm p < 0.05), 'equivalent' (TOST), else 'inconclusive'."""
    if row is None:
        return "missing"
    p = row.get("p_holm", np.nan)
    sig = np.isfinite(p) and p < 0.05
    if sig and row["ci95_lo"] > 0:
        return "higher"
    if sig and row["ci95_hi"] < 0:
        return "lower"
    if row.get("equivalent", False):
        return "equivalent"
    return "inconclusive"


def rq5_reading(T, RT, D, bt):
    """Applies the pre-specified interpretation table (design doc 2.2) per dataset and variant pair."""
    def pick(df, **kw):
        if df is None or not len(df):
            return None
        m = np.ones(len(df), bool)
        for k, v in kw.items():
            m &= (df[k] == v).values
        return df[m].iloc[0].to_dict() if m.any() else None

    rows = []
    for (tag, ds), _ in T.groupby(["tag", "dataset"]):
        for A, B in PAIRS:
            base = dict(tag=tag, dataset=ds, subset="common_correct", A=A, B=B)
            loc = _direction(pick(T, method="gradcam", metric="energy_norm", **base))
            dele = _direction(pick(T, method="gradcam", metric="del_gain", **base))
            ig = _direction(pick(T, method="ig", metric="energy_norm", **base))
            lri = _direction(pick(RT, method="occlusion", metric="lri", **base))
            dod = _direction(pick(D, metric="energy_norm", m1="gradcam", m2="ig", **base))
            if loc == "missing":
                continue
            beats = []
            if bt is not None and len(bt):
                for v in (A, B):
                    try:
                        beats.append(bt.loc[(tag, ds, v, "gradcam"), "energy_norm"] >
                                     bt.loc[(tag, ds, v, "base_center"), "energy_norm"])
                    except KeyError:
                        pass
            if beats and not all(beats):
                reading = "Grad-CAM does not beat base_center for at least one variant: localisation not interpretable here (2.2, row 5)"
            elif loc == "lower" and lri == "equivalent" and dod == "lower" and ig in ("higher", "equivalent"):
                reading = "Method-dependent: Grad-CAM penalises the larger model, the model's lesion reliance does not change (2.2, RQ6 row)"
            elif loc == "lower" and lri == "equivalent" and dele == "lower":
                reading = "Explanation degrades, model does not (2.2, row 1)"
            elif loc == "lower" and lri == "lower":
                reading = "Larger model relies less on the lesion; heatmap reflects that (2.2, row 2)"
            elif loc == "equivalent":
                reading = "Larger model is not more explainable (2.2, row 3)"
            elif loc == "higher":
                reading = "Scaling improves localisation under a locked recipe (2.2, row 4)"
            else:
                reading = "Not conclusive under the pre-specified rules"
            rows.append(dict(tag=tag, dataset=ds, contrast=f"{A.upper()}-{B.upper()}", gradcam_localisation=loc,
                             lesion_reliance=lri, gradcam_deletion=dele, ig_localisation=ig,
                             gradcam_vs_ig_dod=dod, reading=reading))
    return pd.DataFrame(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--bench", default="")
    ap.add_argument("--exclude_rescued", action="store_true",
                    help="sensitivity analysis: drop runs that only succeeded after a rescue attempt")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rng = np.random.default_rng(2026)
    rep = []
    S = load_runs(a.results)
    if S.empty:
        print("no runs found")
        return
    say = lambda title, df: rep.append(f"\n=== {title} ===\n" + (df.to_string() if len(df) else "(empty)"))
    ok = S.status.astype(str).str.startswith(("completed", "early_stop"))
    S["ok"] = ok
    excluded = S.loc[~ok, [c for c in ["tag", "dataset", "variant", "run", "seed", "lr", "attempt", "amp", "status",
                                       "archived"] if c in S]]
    # training stability per variant is itself a result: report it, do not hide it
    fin = S[~S.archived]
    stab = fin.groupby(["tag", "dataset", "variant"]).agg(
        runs=("ok", "size"), ok=("ok", "sum"),
        ok_first_try=("attempt", lambda x: int(((x == 0) & fin.loc[x.index, "ok"]).sum())),
        rescued=("attempt", lambda x: int(((x > 0) & fin.loc[x.index, "ok"]).sum())),
        failed_final=("ok", lambda x: int((~x).sum())))
    arch = S[S.archived].groupby(["tag", "dataset", "variant"]).size().rename("failed_attempts")
    stab = stab.join(arch).fillna({"failed_attempts": 0}).astype({"failed_attempts": int})
    if "bad_loss_frac_total" in S:
        stab = stab.join(fin[fin.ok].groupby(["tag", "dataset", "variant"]).bad_loss_frac_total.max()
                         .rename("max_skipped_batch_frac"))
    stab.to_csv(f"{a.out}/training_stability.csv")
    say("Training stability per variant (rescue protocol: next-lower LR, then FP32)", stab)
    say("Excluded runs (diverged, collapsed, unfinished, or archived failed attempts)", excluded)
    S = S[ok & ~S.archived]
    if a.exclude_rescued:
        S = S[S.attempt == 0]
        say("SENSITIVITY ANALYSIS", pd.DataFrame({"note": ["runs that needed a rescue attempt are excluded"]}))
    S.to_csv(f"{a.out}/runs.csv", index=False)

    desc, comp = perf_tables(S, a.n_boot, rng)
    desc.to_csv(f"{a.out}/perf_descriptive.csv")
    comp.to_csv(f"{a.out}/perf_comparisons.csv", index=False)
    say("Performance (mean, sd, n over repetitions)", desc)
    say("Performance: paired bootstrap differences A - B (Holm within family; equivalent = 90% CI inside margin)",
        comp.round(4) if len(comp) else comp)

    eff_cols = [c for c in ["params_m", "train_sec", "epochs_run", "train_img_per_sec", "energy_kwh", "erf_area90",
                            "xai_sec", "sanity_sec"] if c in S]
    eff = S.groupby(["tag", "dataset", "variant"])[eff_cols].agg(["mean", "std"]).round(3)
    eff.to_csv(f"{a.out}/efficiency_and_erf_from_runs.csv")
    say("Run-level efficiency (confounded by parallel runs; see bench) and ERF area (descriptive)", eff)
    if a.bench and os.path.exists(a.bench):
        b = pd.DataFrame(json.load(open(a.bench)))
        b.to_csv(f"{a.out}/efficiency_benchmark.csv", index=False)
        say("Controlled efficiency benchmark", b.round(3))

    T = RT = D = bt = None
    X = load_per_image(S, "xai_per_image.csv")
    if len(X):
        X = add_size_tertiles(add_derived(X))
        X.to_csv(f"{a.out}/xai_per_image_all.csv", index=False)
        bt = baseline_table(X)
        bt.to_csv(f"{a.out}/xai_vs_baselines.csv")
        say("Explanation scores vs reference maps (base_center, base_random, oracle_grid)", bt)
        XC = common_correct(X)
        metrics = [m for m in ["energy_norm", "hit", "iou_topk", "rra", "del_gain", "ins_gain"] if m in X]
        methods = [m for m in X.method.unique() if not m.startswith(("base_", "oracle"))]
        T = pd.concat([paired_tests(XC, metrics, methods, a.n_boot, rng, "common_correct"),
                       paired_tests(X, metrics, methods, a.n_boot, rng, "all_lesion_images")], ignore_index=True)
        if len(T):
            prim = (T.method == PRIMARY_XAI[0]) & (T.metric == PRIMARY_XAI[1]) & (T.subset == "common_correct")
            T["family"] = np.where(prim, "primary_xai", np.where(T.subset == "common_correct", "secondary_xai", "sensitivity"))
            T["p_holm"] = np.nan
            for (t, fam), idx in T.groupby(["tag", "family"]).groups.items():
                T.loc[idx, "p_holm"] = holm(T.loc[idx, "p_wilcoxon"])
            T.to_csv(f"{a.out}/xai_paired_tests.csv", index=False)
            say("PRIMARY endpoint: Grad-CAM energy_norm, images correct for all variants",
                T[T.family == "primary_xai"].round(4))
            say("Secondary explanation endpoints", T[T.family == "secondary_xai"].round(4))
        D = pd.concat([method_dependence(XC, a.n_boot, rng, "common_correct"),
                       method_dependence(X, a.n_boot, rng, "all_lesion_images")], ignore_index=True)
        if len(D):
            D["p_holm"] = np.nan
            for (t, fam), idx in D.groupby(["tag", "family"]).groups.items():
                D.loc[idx, "p_holm"] = holm(D.loc[idx, "p_wilcoxon"])
            D.to_csv(f"{a.out}/rq6_method_dependence.csv", index=False)
            say("RQ6: method dependence, dod = (A-B)_m1 - (A-B)_m2 (family rq6 = Grad-CAM vs IG, energy_norm, common-correct)",
                D[D.family == "rq6"].round(4))
        if "rra_derived" in X and X.rra_derived.any():
            say("Note", pd.DataFrame({"note": [f"RRA derived from iou_topk for {int(X.rra_derived.sum())} rows (runs made before RRA was computed directly)"]}))
        lmm = lesion_size_lmm(XC)
        lmm.to_csv(f"{a.out}/h4_lesion_size_lmm.csv", index=False)
        say("H4: variant x lesion-size interaction (LMM, LRT)", lmm.drop(columns=["coefs"], errors="ignore"))

    R = load_per_image(S, "reliance_per_image.csv")
    if len(R):
        R["method"] = "occlusion"
        R.to_csv(f"{a.out}/reliance_per_image_all.csv", index=False)
        rt = R.groupby(["tag", "dataset", "variant"])[["drop_lesion", "drop_ctrl", "lri", "p_lesion_only", "ctrl_overlap"]] \
            .mean().round(4)
        rt.to_csv(f"{a.out}/reliance_summary.csv")
        say("Lesion reliance (model property, no XAI)", rt)
        RT = pd.concat([paired_tests(common_correct(R), ["lri", "drop_lesion"], ["occlusion"], a.n_boot, rng, "common_correct"),
                        paired_tests(R, ["lri", "drop_lesion"], ["occlusion"], a.n_boot, rng, "all_lesion_images")])
        if len(RT):
            RT["p_holm"] = holm(RT.p_wilcoxon)
            RT.to_csv(f"{a.out}/reliance_paired_tests.csv", index=False)
            say("Lesion reliance: paired differences", RT.round(4))

    if T is not None and len(T):
        RD = rq5_reading(T, RT, D, bt)
        RD.to_csv(f"{a.out}/rq5_rq6_reading.csv", index=False)
        say("RQ5/RQ6 reading by the pre-specified interpretation table (design doc 2.2)", RD)

    SN = load_per_image(S, "sanity_per_image.csv")
    if len(SN):
        st = SN.groupby(["tag", "dataset", "variant", "method", "level", "level_name"]).rank_corr.mean() \
            .unstack(["level", "level_name"]).round(3)
        st.to_csv(f"{a.out}/sanity_cascade.csv")
        say("Sanity check: mean Spearman(trained map, randomized map) by cascade level (lower = passes)", st)

    open(f"{a.out}/analysis_report.txt", "w").write("\n".join(rep))
    print("\n".join(rep))


if __name__ == "__main__":
    main()

## 3. Deteksi path dataset dan pemulihan hasil sesi sebelumnya

In [ ]:
import glob, os, shutil, json, time, fnmatch
from pathlib import Path

def _walk_find(pattern, root=INPUT_ROOT, max_depth=6):
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        for f in filenames:
            if fnmatch.fnmatch(f, pattern):
                return dirpath
    return None

def _top(p, root=INPUT_ROOT):
    # folder dataset tingkat atas (menangani /kaggle/input/<slug> dan /kaggle/input/datasets/<owner>/<slug>)
    parts = Path(p).relative_to(root).parts
    depth = 3 if parts[0] == "datasets" and len(parts) >= 3 else 1
    return str(Path(root, *parts[:depth]))

paths = dict(DATASET_PATHS)
if paths["ham10000_images"] is None:
    d = _walk_find("HAM10000_metadata*")
    paths["ham10000_images"] = _top(d) if d else None
if paths["ham10000_masks"] is None:
    d = _walk_find("ISIC_*_segmentation.png")
    paths["ham10000_masks"] = _top(d) if d else None
if paths["siim"] is None:
    for d in glob.glob(f"{INPUT_ROOT}/*") + glob.glob(f"{INPUT_ROOT}/datasets/*/*"):
        if "pneumo" in d.lower() or "siim" in d.lower():
            paths["siim"] = d
            break
if paths["busi"] is None:
    for d in glob.glob(f"{INPUT_ROOT}/*") + glob.glob(f"{INPUT_ROOT}/datasets/*/*"):
        if "busi" in d.lower() or "breast" in d.lower():
            paths["busi"] = d
            break
    if paths["busi"] is None:
        d = _walk_find("malignant*")
        paths["busi"] = _top(d) if d else None
for k, v in paths.items():
    print(f"{k:16s} -> {v}")

# lanjutkan hasil dari sesi sebelumnya (output notebook lama yang ditambahkan sebagai input)
os.makedirs(RESULTS, exist_ok=True)
for prev in glob.glob(f"{INPUT_ROOT}/**/results_v3", recursive=True):
    if os.path.isdir(prev) and os.path.commonpath([os.path.abspath(prev), WORK]) != WORK:
        shutil.copytree(prev, RESULTS, dirs_exist_ok=True)
        print("restored results from", prev)
os.makedirs(RESULTS_LR, exist_ok=True)
for prev in glob.glob(f"{INPUT_ROOT}/**/results_lr_v3", recursive=True):
    if os.path.isdir(prev) and os.path.commonpath([os.path.abspath(prev), WORK]) != WORK:
        shutil.copytree(prev, RESULTS_LR, dirs_exist_ok=True)
        print("restored lr_search results from", prev)
for prev in glob.glob(f"{INPUT_ROOT}/**/best_lr.json", recursive=True):
    shutil.copy(prev, f"{WORK}/best_lr.json")
    print("restored", prev)
print("DONE runs available:", len(glob.glob(f"{RESULTS}/**/DONE", recursive=True)),
      "| lr_search runs:", len(glob.glob(f"{RESULTS_LR}/**/DONE", recursive=True)))

## 4. Siapkan dataset (indeks, split, cache per resolusi) dan unduh bobot sekali

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, WORK)
import importlib, data_utils
importlib.reload(data_utils)
from train import WEIGHTS

plan = PLANS[MODE]
sizes = sorted(set(plan["sizes"][v] for v in plan["variants"]))
# unduh bobot sekali di sini, supaya dua proses paralel tidak menulis cache Hugging Face yang sama
if "--no_pretrained" not in EXTRA_TRAIN_ARGS:
    for v in plan["variants"]:
        m = timm.create_model(WEIGHTS[v], pretrained=True)
        print(WEIGHTS[v], "mean/std", m.pretrained_cfg["mean"], m.pretrained_cfg["std"])
        del m
prepared = {}
# v3.4: dengan SHARD, siapkan hanya dataset milik shard ini (akun lain tidak perlu input dataset yang tidak dipakai)
for ds in [d for d in plan["datasets"] if MODE != "full" or SHARD is None or d in SHARDS[SHARD]]:
    t = time.time()
    df, splits = data_utils.prepare(ds, paths, CACHE, sizes, workers=4, siim_neg_ratio=SIIM_NEG_PER_POS)
    prepared[ds] = splits
    for s in sizes:
        ms = pd.read_csv(f"{CACHE}/{ds}/{s}/masks.csv")
        print(f"{ds} @{s}px: {len(df)} images, {int((ms.mask_area > 0).sum())} lesion masks, "
              f"{int(ms.mask_lost.sum())} masks lost by resizing, median lesion area {ms.mask_area.median():.4f}")
    print(f"   splits={ {k: {s: len(v[s]) for s in v} for k, v in splits.items()} } ({time.time()-t:.0f}s)")

## 5. Jalankan run (dua proses paralel, satu per GPU T4)

In [ ]:
import subprocess, threading, sys

N_GPU = max(1, torch.cuda.device_count())
best_lr = json.load(open(f"{WORK}/best_lr.json")) if os.path.exists(f"{WORK}/best_lr.json") else {}
COST = {"s": 1, "m": 2, "l": 4}
SIZE = {"busi": 0.05, "ham10000": 1.0, "siim": 0.55}

def lr_key(ds, v, run):
    return f"{ds}/{v}/{run}"            # BUSI: per fold; HAM10000/SIIM: run = "fixed"

def build_jobs(mode):
    p, jobs = PLANS[mode], []
    for ds in ([] if mode == "lr_extend" else p["datasets"]):   # lr_extend: jobs come from extension_jobs()
        runs = [(str(f), f) for f in p.get("busi_folds", [0])] if ds == "busi" else \
            [("fixed", s) for s in p.get("seeds", [1])]
        for v in p["variants"]:
            size = p["sizes"][v]
            if mode == "lr_search":
                for run, seed in (runs if ds == "busi" else runs[:1]):
                    for lr in p["lrs"]:
                        jobs.append(dict(ds=ds, v=v, run=run, seed=seed, lr=lr, epochs=p["epochs"], frac=1.0,
                                         size=size, xai=False, tag="lr",
                                         out=f"{RESULTS_LR}/{ds}/{v}/run-{run}/lr-{lr:g}"))
                continue
            for frac in p.get("fracs", [1.0]):
                for run, seed in runs:
                    name = f"run-{run}_seed-{seed}_{size}px" + (f"_frac-{frac:g}" if frac < 1 else "")
                    jobs.append(dict(ds=ds, v=v, run=run, seed=seed, frac=frac, xai=True, size=size, tag=p["tag"],
                                     lr=best_lr.get(lr_key(ds, v, run), DEFAULT_LR), epochs=p["epochs"][ds],
                                     out=f"{RESULTS}/{p['tag']}/{ds}/{v}/{name}"))
    if mode == "lr_extend":
        jobs = extension_jobs(p)
    if mode == "full" and SHARD is not None:
        jobs = [j for j in jobs if shard_of(j) == SHARD]
    for j in jobs:
        j["cost"] = COST[j["v"]] * SIZE[j["ds"]] * j["epochs"] * j["frac"] * (j["size"] / 384) ** 2
    return sorted(jobs, key=lambda j: -j["cost"])   # terpanjang dulu agar dua GPU seimbang

def build_jobs_unsharded(mode):
    global SHARD
    saved, SHARD = SHARD, None
    try:
        return build_jobs(mode)
    finally:
        SHARD = saved


def shard_of(j):
    """Shard that owns a full-plan job: BUSI by fold, HAM10000/SIIM by seed (see SHARDS in the config cell)."""
    key = int(j["run"]) if j["ds"] == "busi" else int(j["seed"])
    owners = [k for k, spec in SHARDS.items() if key in spec.get(j["ds"], [])]
    return owners[0] if len(owners) == 1 else None if not owners else tuple(owners)


# rough GPU minutes per full run (training + XAI), calibrated on the lr_search session (T4, 384 px)
EST_MIN = {("ham10000", "s"): 81, ("ham10000", "m"): 122, ("ham10000", "l"): 220,
           ("siim", "s"): 50, ("siim", "m"): 82, ("siim", "l"): 142,
           ("busi", "s"): 15, ("busi", "m"): 21, ("busi", "l"): 31}


def lr_table():
    """All lr_search / lr_extend runs found in RESULTS_LR (restored from the input)."""
    rows = []
    for f in glob.glob(f"{RESULTS_LR}/*/*/run-*/lr-*/summary.json"):
        d = json.load(open(f))
        rows.append(dict(ds=d["dataset"], v=d["variant"], run=str(d["run"]), seed=int(d["seed"]), lr=float(d["lr"]),
                         val_f1=d["best_val_macro_f1"], status=d["status"],
                         skipped=d.get("bad_loss_frac_total", float("nan"))))
    df = pd.DataFrame(rows)
    if len(df):
        df["ok"] = df.status.str.startswith(("completed", "early_stop"))
    return df


def extension_jobs(p):
    """Pre-specified one-step extension: best stable LR at the lower edge of the base grid -> also try ext_low;
    at the upper edge -> also try ext_high; no stable LR at all -> try ext_low."""
    df = lr_table()
    if df.empty:
        raise RuntimeError("lr_extend butuh hasil lr_search: tambahkan output versi lr_search sebagai input "
                           "(folder results_lr_v3 harus terpulihkan di sel 3).")
    base = sorted(PLANS["lr_search"]["lrs"])
    df = df[df.lr.isin(base)]
    jobs = []
    for (ds, v, run), g in df.groupby(["ds", "v", "run"]):
        if ds not in p["datasets"] or v not in p["variants"]:
            continue
        ok = g[g.ok]
        if ok.empty:
            new = p["ext_low"]
        else:
            best = float(ok.loc[ok.val_f1.idxmax(), "lr"])
            new = p["ext_low"] if best == base[0] else p["ext_high"] if best == base[-1] else None
        if new is None:
            continue
        jobs.append(dict(ds=ds, v=v, run=run, seed=int(g.seed.iloc[0]), lr=new, epochs=p["epochs"], frac=1.0,
                         size=p["sizes"][v], xai=False, tag="lr", out=f"{RESULTS_LR}/{ds}/{v}/run-{run}/lr-{new:g}"))
    return jobs


def cmd_for(j):
    p = PLANS[MODE]
    c = [sys.executable, f"{WORK}/train.py", "--dataset", j["ds"], "--variant", j["v"], "--run", j["run"],
         "--seed", str(j["seed"]), "--cache", CACHE, "--out", j["out"], "--img_size", str(j["size"]),
         "--epochs", str(j["epochs"]), "--patience", str(PATIENCE), "--lr", str(j["lr"]), "--bn_mode", BN_MODE,
         "--train_frac", str(j["frac"]), "--workers", str(WORKERS_PER_RUN), "--tag", j["tag"],
         "--deadline", str(T0 + HARD_DEADLINE_H * 3600), "--attempt", str(j.get("attempt", 0)),
         "--shard", "" if SHARD is None else str(SHARD)]
    if j.get("no_amp"):
        c += ["--no_amp"]
    if not j["xai"]:
        c += ["--no_xai"]
    else:
        for k in ("xai_n", "insdel_n", "reliance_n", "sanity_n", "scorecam_n"):
            c += [f"--{k}", str(p[k])]
    if KEEP_CKPT:
        c += ["--keep_ckpt"]
    return c + list(EXTRA_TRAIN_ARGS)

if MODE == "full" and SHARD is not None:
    _all = [j for j in build_jobs_unsharded("full")]
    _bad = [(j["ds"], j["v"], j["run"], j["seed"], shard_of(j)) for j in _all if not isinstance(shard_of(j), int)]
    assert not _bad, f"SHARDS tidak mencakup setiap run tepat satu kali: {_bad[:5]}"
    print(f"SHARD {SHARD}: {sum(shard_of(j) == SHARD for j in _all)} dari {len(_all)} run plan full; pembagian lengkap dan tidak tumpang tindih.")

T0 = time.time()
queue, lock = build_jobs(MODE), threading.Lock()
if MODE == "full":
    _gpu_h = sum(EST_MIN.get((j["ds"], j["v"]), 60) for j in queue) / 60
    print(f"Perkiraan beban plan ini{'' if SHARD is None else f' (SHARD {SHARD})'}: {_gpu_h:.1f} jam GPU "
          f"= sekitar {_gpu_h / max(N_GPU, 1):.1f} jam sesi dengan {N_GPU} GPU (sebelum early stopping)")
todo = [j for j in queue if not os.path.exists(f"{j['out']}/DONE")]
print(f"{len(queue)} jobs in plan '{MODE}', {len(todo)} not done yet, {N_GPU} GPU(s)")
if MODE not in ("lr_search", "lr_extend"):
    missing = sorted({lr_key(j["ds"], j["v"], j["run"]) for j in todo} - set(best_lr))
    if missing:
        print(f"PERINGATAN: {len(missing)} kombinasi belum punya LR hasil lr_search, memakai DEFAULT_LR={DEFAULT_LR}:",
              missing[:6], "...")
queue = todo

LR_GRID = sorted(set(PLANS["lr_search"]["lrs"]) | {PLANS["lr_extend"]["ext_low"], PLANS["lr_extend"]["ext_high"]})
USE_RESCUE = MODE in RESCUE_MODES


def lower_lr(lr):
    lower = [x for x in LR_GRID if x < lr * 0.999]
    return max(lower) if lower else lr / 3


def run_status(out):
    f = f"{out}/summary.json"
    return json.load(open(f))["status"] if os.path.exists(f) else None


def is_failed(st):
    return st is not None and st.startswith(("diverged", "collapsed"))


def attempt_plan(j, k):
    """k = 0: planned run; 1: same seed/split, next-lower LR; 2: that LR in FP32."""
    jj = dict(j, attempt=k, no_amp=False)
    if k >= 1:
        jj["lr"] = lower_lr(j["lr"])
    if k >= 2:
        jj["no_amp"] = True
    return jj


def run_once(j, gpu):
    os.makedirs(j["out"], exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PHYSICAL_GPU=str(gpu), PYTHONPATH=WORK, TQDM_DISABLE="1")
    t = time.time()
    with open(f"{j['out']}/log.txt", "w") as f:
        rc = subprocess.run(cmd_for(j), env=env, stdout=f, stderr=subprocess.STDOUT, cwd=WORK).returncode
    return run_status(j["out"]), rc, (time.time() - t) / 60


def worker(gpu):
    while True:
        with lock:
            if not queue:
                return
            if time.time() - T0 > STOP_LAUNCH_AFTER_H * 3600:
                print(f"[gpu{gpu}] session time budget reached, not starting new jobs")
                return
            j = queue.pop(0)
        name = j["out"].replace(WORK + "/", "")
        k0 = len(glob.glob(f"{j['out']}__failed-attempt*"))   # attempts archived in earlier sessions
        last = MAX_RESCUE if USE_RESCUE else 0
        for k in range(k0, last + 1):
            if k > k0 and time.time() - T0 > STOP_LAUNCH_AFTER_H * 3600:
                print(f"[gpu{gpu}] no time left for rescue attempt {k} of {name}; it continues next session")
                return
            jj = attempt_plan(j, k)
            print(f"[gpu{gpu}] start {name} (attempt {k}, lr {jj['lr']:g}{', FP32' if jj['no_amp'] else ''})", flush=True)
            st, rc, mins = run_once(jj, gpu)
            if st is None or not os.path.exists(f"{j['out']}/DONE"):
                print(f"[gpu{gpu}] NOT DONE (rc={rc}, see log.txt) {name} in {mins:.1f} min", flush=True)
                break
            print(f"[gpu{gpu}] {st} {name} in {mins:.1f} min", flush=True)
            if not is_failed(st) or k == last:
                break
            shutil.move(j["out"], f"{j['out']}__failed-attempt{k}")
            print(f"[gpu{gpu}] RESCUE: {name} failed ({st}); archived as __failed-attempt{k}", flush=True)


threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
[t.start() for t in threads]
[t.join() for t in threads]
print(f"session finished after {(time.time()-T0)/3600:.2f} h; remaining jobs: {len(queue)}")

# ringkasan status run pada plan ini (diverged/collapsed = gagal, tidak ikut dianalisis)
rows = []
for j in build_jobs(MODE):
    f = f"{j['out']}/summary.json"
    if os.path.exists(f):
        d = json.load(open(f))
        rows.append(dict(run=j["out"].replace(WORK + "/", ""), variant=j["v"], lr=d["lr"], attempt=d.get("attempt", 0),
                         fp32=not d.get("amp", True), status=d["status"], epochs=d["epochs_run"],
                         skipped_batches=round(d.get("bad_loss_frac_total", float("nan")), 4),
                         failed_attempts=len(glob.glob(f"{j['out']}__failed-attempt*")),
                         val_f1=round(d["best_val_macro_f1"], 4), test_f1=round(d["test_macro_f1"], 4)))
if rows:
    st = pd.DataFrame(rows)
    display(st)
    st["ok"] = ~st.status.str.startswith(("diverged", "collapsed"))
    print("Ringkasan per varian (laporkan di naskah):")
    st["rescued"] = (st.attempt > 0) & st.ok
    display(st.groupby("variant").agg(runs=("ok", "size"), ok=("ok", "sum"), rescued=("rescued", "sum"),
                                      failed_attempts=("failed_attempts", "sum"), max_skipped=("skipped_batches", "max")))
    bad = st[~st.ok]
    if len(bad):
        if USE_RESCUE:
            print(f"PERINGATAN: {len(bad)} run tetap gagal setelah protokol penyelamatan; dikeluarkan dan dilaporkan.")
        else:
            print(f"PERINGATAN: {len(bad)} run gagal (divergen/kolaps). Cek log.txt dan history.csv "
                  "(kolom bad_loss_frac, val_n_pred_classes). Di pilot: coba BN_MODE='train' untuk satu run L di BUSI "
                  "fold 0 sebelum menurunkan LR atau memakai FP32.")

## 6. Pilih learning rate (khusus mode `lr_search`)

In [ ]:
# LR terbaik per dataset/varian (dan per fold untuk BUSI) berdasarkan val macro-F1.
# Data validasi BUSI fold k berada di dalam bagian train fold k, jadi test fold k tidak pernah dipakai memilih LR.
# Hanya run yang stabil (completed/early_stop) yang boleh terpilih: run yang divergen/kolaps bisa saja sempat
# punya val F1 tinggi sebelum rusak, dan LR itu akan gagal lagi di plan full.
# Setelah lr_extend, pemilihan memakai semua LR yang diuji (grid awal + perluasan).
if MODE in ("lr_search", "lr_extend"):
    lr_df = lr_table()
    print("val macro-F1 (run gagal = NaN):")
    display(lr_df.assign(val_f1=lr_df.val_f1.where(lr_df.ok)).pivot_table(
        index=["ds", "v", "run"], columns="lr", values="val_f1").round(4))
    fails = lr_df[~lr_df.ok]
    if len(fails):
        print(f"{len(fails)} run gagal (dilaporkan sebagai hasil stabilitas):")
        display(fails.groupby(["ds", "v", "lr"]).size().rename("failed_runs").reset_index())
    okdf = lr_df[lr_df.ok]
    best = okdf.loc[okdf.groupby(["ds", "v", "run"]).val_f1.idxmax()].copy()
    best_lr = {f"{r.ds}/{r.v}/{r.run}": float(r.lr) for r in best.itertuples()}
    no_stable = sorted(set(lr_df[["ds", "v", "run"]].astype(str).agg("/".join, axis=1)) - set(best_lr))
    if no_stable:
        print("PERINGATAN: tidak ada LR yang stabil untuk", no_stable,
              "-> kombinasi ini TIDAK diberi LR (plan full akan memakai DEFAULT_LR dan memberi peringatan).")
    json.dump(best_lr, open(f"{WORK}/best_lr.json", "w"), indent=2)
    print(best_lr)
    # tepi grid dihitung per kombinasi, terhadap LR yang benar-benar diuji untuk kombinasi itu
    tested = lr_df.groupby(["ds", "v", "run"]).lr.agg(["min", "max", "count"])
    best = best.join(tested, on=["ds", "v", "run"])
    best["at_edge"] = (best.lr == best["min"]) | (best.lr == best["max"])
    best["extended"] = best["count"] > len(PLANS["lr_search"]["lrs"])
    best[["ds", "v", "run", "lr", "val_f1", "count", "extended", "at_edge"]] \
        .rename(columns={"count": "n_lr_tested"}).to_csv(f"{WORK}/lr_selection.csv", index=False)
    edge = best[best.at_edge]
    if len(edge):
        if MODE == "lr_search":
            print("LR terbaik berada di tepi grid untuk", len(edge), "kombinasi -> jalankan MODE = 'lr_extend':",
                  edge[["ds", "v", "run", "lr"]].values.tolist())
        else:
            print("Masih di tepi setelah perluasan (dipakai apa adanya dan dilaporkan; tidak diperluas lagi):",
                  edge[edge.extended][["ds", "v", "run", "lr"]].values.tolist())
    print("Ringkasan pemilihan LR tersimpan di lr_selection.csv")


## 7. Benchmark efisiensi terkontrol (sekali, setelah training, satu proses di GPU 0)

In [ ]:
import pandas as pd
if MODE in ("pilot", "full", "native_res") and SHARD is None:   # v3.4: benchmark hanya di sesi penggabungan akhir
    variants = ",".join(PLANS[MODE]["variants"])
    bench_sizes = ",".join(str(s) for s in sorted({IMG_SIZE, *PLANS[MODE]["sizes"].values(), *NATIVE.values()}))
    subprocess.run([sys.executable, f"{WORK}/bench.py", "--variants", variants, "--img_sizes", bench_sizes,
                    "--out", f"{WORK}/efficiency_benchmark.json"], env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), check=True)
    display(pd.DataFrame(json.load(open(f"{WORK}/efficiency_benchmark.json"))).round(2))

## 8. Analisis statistik (endpoint primer, uji berpasangan, TOST, Holm, LMM untuk H4, RQ6, dan bacaan tabel 2.2)
Keluaran di `/kaggle/working/summary/`:
`perf_comparisons.csv`, `xai_paired_tests.csv` (keluarga `primary_xai`, `secondary_xai`, `sensitivity`), `xai_vs_baselines.csv`,
`h4_lesion_size_lmm.csv`, `rq6_method_dependence.csv`, `rq5_rq6_reading.csv` (bacaan otomatis menurut tabel interpretasi 2.2), `reliance_*.csv`, `sanity_cascade.csv`, `efficiency_*.csv`, dan ringkasan `analysis_report.txt`.
Pada pilot hasilnya belum bermakna secara statistik; pakai hanya untuk memeriksa pipeline dan waktu.

In [ ]:
os.makedirs(f"{WORK}/summary", exist_ok=True)
def run_analysis(out, extra=()):
    r = subprocess.run([sys.executable, f"{WORK}/analysis.py", "--results", RESULTS, "--out", out,
                        "--bench", f"{WORK}/efficiency_benchmark.json", "--n_boot", "2000" if MODE == "full" else "300",
                        *extra], env=dict(os.environ, PYTHONPATH=WORK), capture_output=True, text=True)
    print(r.stdout)   # v3.2: laporan utuh (v3.1 hanya mencetak 20.000 karakter terakhir)
    if r.returncode:
        print(r.stderr[-5000:])

run_analysis(f"{WORK}/summary")
# analisis sensitivitas: tanpa run yang baru berhasil setelah protokol penyelamatan
_runs = pd.read_csv(f"{WORK}/summary/runs.csv") if os.path.exists(f"{WORK}/summary/runs.csv") else pd.DataFrame()
if len(_runs) and "attempt" in _runs and (_runs.attempt > 0).any():
    print("\n\n########## SENSITIVITY: tanpa run yang diselamatkan ##########")
    run_analysis(f"{WORK}/summary_no_rescued", ["--exclude_rescued"])


## 9. Gambar deskriptif: ERF rata-rata per varian

In [ ]:
import numpy as np, matplotlib.pyplot as plt
os.makedirs(f"{WORK}/summary/figs", exist_ok=True)
S = pd.read_csv(f"{WORK}/summary/runs.csv") if os.path.exists(f"{WORK}/summary/runs.csv") else pd.DataFrame()
for (tag, ds), g in (S.groupby(["tag", "dataset"]) if len(S) else []):
    vs = [v for v in ["s", "m", "l"] if v in set(g.variant)]
    fig, axes = plt.subplots(1, len(vs), figsize=(3.2 * len(vs), 3.2), squeeze=False)
    for ax, v in zip(axes[0], vs):
        maps = [np.load(f"{d}/erf.npy") for d in g[g.variant == v].dir if os.path.exists(f"{d}/erf.npy")]
        if maps:
            e = np.mean(maps, 0)
            ax.imshow(np.log1p(e / e.max() * 100), cmap="magma")
            ax.set_title(f"{v.upper()}  area90={g[g.variant == v].erf_area90.mean():.2f}")
        ax.axis("off")
    fig.suptitle(f"Effective receptive field ({tag}, {ds}), log scale")
    fig.tight_layout()
    fig.savefig(f"{WORK}/summary/figs/erf_{tag}_{ds}.png", dpi=150)
    plt.show()

## 10. Kalibrasi waktu dari pilot (proyeksi jam GPU untuk plan `full`)

In [ ]:
if len(S):
    thr = S.groupby(["dataset", "variant"]).train_img_per_sec.mean()
    print("Throughput training (citra/detik, termasuk gangguan run paralel):\n", thr.round(1))
    xai_min = (S.groupby(["dataset", "variant"])[["xai_sec", "sanity_sec"]].mean() / 60).round(1)
    print("\nMenit XAI + sanity per run (pilot memakai subset kecil; skala linear ke jumlah citra full):\n", xai_min)
    full, est = PLANS["full"], 0.0
    for ds in full["datasets"]:
        if not os.path.exists(f"{CACHE}/{ds}/splits.json"):
            continue
        sp = json.load(open(f"{CACHE}/{ds}/splits.json"))
        n_tr = len(next(iter(sp.values()))["train"])
        reps = len(full["busi_folds"]) if ds == "busi" else len(full["seeds"])
        for v in full["variants"]:
            ips = thr.get((ds, v), np.nan)
            if np.isnan(ips) and (ds, "s") in thr and (ds, "l") in thr:
                ips = 2 / (1 / thr[(ds, "s")] + 1 / thr[(ds, "l")])   # M: kira-kira di antara S dan L
            if not np.isnan(ips):
                est += reps * full["epochs"][ds] * n_tr / ips / 3600
    print(f"\nPerkiraan training plan 'full' (batas atas, tanpa early stopping): {est:.1f} jam GPU "
          f"= {est/2:.1f} jam kuota dengan 2 GPU paralel (belum termasuk validasi dan XAI)")

## Langkah berikutnya
1. **Pilot** → pastikan tabel status di sel 5 tidak berisi `diverged`/`collapsed`, `skipped_batches` = 0, dan val F1 L naik seperti S (kalau L masih gagal: coba `BN_MODE="train"` untuk satu run L di BUSI fold 0 dulu); cek `summary/analysis_report.txt`, jumlah masker yang hilang saat resize, `xai_vs_baselines.csv` (apakah Grad-CAM mengalahkan `base_center`?), proyeksi jam GPU, dan beberapa `log.txt`.
2. **Kunci rencana analisis** (margin, endpoint primer) dan pra-registrasi di OSF.
3. **lr_search**, lalu **lr_extend** (input: output versi lr_search) → `best_lr.json` (dipakai otomatis oleh `full` dan `native_res`); hanya LR dari run yang stabil yang bisa terpilih. Kalau sesi terpotong, jadikan output sebagai input: `results_lr_v3/` ikut dipulihkan.
4. **full** (satu akun, atau dibagi ke 3 akun milik 3 orang dengan `SHARD` = 1/2/3, lalu satu sesi penggabungan `SHARD = None` di akun utama) → jalankan berulang di beberapa sesi (setiap commit melanjutkan run yang belum `DONE`).
5. **native_res** → analisis sensitivitas resolusi.
6. Jalankan ulang sel 7–9 setelah semua run selesai, lalu tulis naskah dari `summary/` (termasuk `training_stability.csv`; bila ada run yang diselamatkan, bandingkan dengan `summary_no_rescued/`).